# Unit tests
This notebook is designed to test the environment component by component and . *Run all* should not lead to any error (and take a couple of minutes).

# Config

In [ ]:
from utils import sample_init_parameters, check_genset_running, generate_init_parameters
import yaml
import os
import copy


Check loading and random parameters

In [ ]:
config = yaml.safe_load(open("..\\configs\\config.yaml", "r")) 


config_microgrid = config["environment"]["microgrid"]

assert config_microgrid["device"]['init_params']["genset_group"]["device"]['init_params']["gensets"][0]["controller"]["init_params"]["status"]["value"] == "random_list"
assert config_microgrid["device"]['init_params']["external_world"]["init_date_time"]["value"] == "random"



Check that random parameters are initialized by set_init_parameters

In [ ]:
config_microgrid_sampled = copy.deepcopy(config_microgrid)
config_microgrid_sampled = sample_init_parameters(config_microgrid_sampled)

assert config_microgrid_sampled["device"]['init_params']["genset_group"]["device"]['init_params']["gensets"][0]["controller"]["init_params"]["status"]["value"] in ["running", "off", "warmup", "cooldown"]
assert config_microgrid_sampled["device"]['init_params']["external_world"]["init_date_time"]["value"] != "random"


Check that check_genset_running corrects non-compatible initializations between gensets and genset controllers

In [ ]:
config_microgrid_sampled["device"]['init_params']["genset_group"]["device"]['init_params']["gensets"][0]["device"]["init_params"]["running"]["value"] = False
config_microgrid_sampled["device"]['init_params']["genset_group"]["device"]['init_params']["gensets"][0]["controller"]["init_params"]["status"]["value"] = "running" # This should be corrected

config_microgrid_sampled = check_genset_running(config_microgrid_sampled)
assert config_microgrid_sampled["device"]['init_params']["genset_group"]["device"]['init_params']["gensets"][0]["controller"]["init_params"]["status"]["value"] == "off" # Check that it is corrected




In [ ]:
config_microgrid_sampled["device"]['init_params']["genset_group"]["device"]['init_params']["gensets"][0]["device"]["init_params"]["running"]["value"] = True
config_microgrid_sampled["device"]['init_params']["genset_group"]["device"]['init_params']["gensets"][0]["controller"]["init_params"]["status"]["value"] = "off" # This should be corrected

config_microgrid_sampled = check_genset_running(config_microgrid_sampled)
assert config_microgrid_sampled["device"]['init_params']["genset_group"]["device"]['init_params']["gensets"][0]["controller"]["init_params"]["status"]["value"] in ["running", "warmup", "cooldown"] # Check that it is corrected


Put them together and check that over 100 tries it never leads to a non-compatible set of parameters

In [ ]:
for i in range(100):
    config_microgrid_sampled = copy.deepcopy(config_microgrid)
    config_microgrid_generated = generate_init_parameters(config_microgrid_sampled)
    assert config_microgrid_generated["device"]['init_params']["genset_group"]["device"]['init_params']["gensets"][0]["controller"]["init_params"]["status"]["value"] in ["running", "off", "warmup", "cooldown"]
    assert config_microgrid_generated["device"]['init_params']["external_world"]["init_date_time"]["value"] != "random"
    genset_0_controller_status = config_microgrid_generated["device"]['init_params']["genset_group"]["device"]['init_params']["gensets"][0]["controller"]["init_params"]["status"]["value"]
    genset_0_running = config_microgrid_generated["device"]['init_params']["genset_group"]["device"]['init_params']["gensets"][0]["device"]["init_params"]["running"]["value"]
    assert (genset_0_running and (genset_0_controller_status in ["running", "warmup", "cooldown"])) or (not genset_0_running and (genset_0_controller_status == "off"))

# Genset

In [ ]:
from genset import Genset
from genset_controller import GensetController
import numpy as np

In [ ]:
genset_params = {
    'device': {
        'const_params': {
            'alpha_g': {
                'value': 0.25,                # in l/kWh
                'type': 'float',
            },
            'beta_g': {
                'value': 10,                  # in l/h
                'type': 'float',
            },
            'prime_power_rating': {
                'value': 400,                 # in kW
                'type': 'float',
            },
            'temp_overload_factor': {
                'value': 1.1,                 # ratio (1.1 for 10% overload)
                'type': 'float',
            },
            'time_step': {
                'value': 1,                   # in minutes
                'type': 'int',
            },
            'noise_level': {
                'value': 'none',              # options: {none, low, medium, high}
                'type': 'string',
            },
            'noise_range_percentage': {
                'value': 0.2,
                'type': 'float',
            },
        },
        'init_params': {
            'running': {
                'value': False,
                'type': 'bool',
            },
            'init_power_setpoint': {
                'value': 300,
                'type': 'float',
            },
        }
    },
    'controller': {
        'const_params': {
            'minimum_run_time': {
                'value': 30,
                'type': 'integer',
            },
            'minimum_load': {
                'value': 120,
                'type': 'float',
            },
            'warmup_power': {
                'value': 100,
                'type': 'float',
            },
            'cooldown_power': {
                'value': 0,
                'type': 'float',
            },
            'warmup_time': {
                'value': 3,
                'type': 'integer',
            },
            'cooldown_time': {
                'value': 6,
                'type': 'integer',
            },
            'average_active_power_constraint': {
                'value': True,
                'type': 'boolean',
            },
            'average_active_power_window': {
                'value': 2880,
                'type': 'integer',
            },
            'average_active_power_limit': {
                'value': 0.7,
                'type': 'float',
            },
        },
        'init_params': {
            'status': {
                'value': 'off',
                'type': 'string',
                'elements': ['off', 'running', 'warmup', 'cooldown'],
            },
            'time_since_warmup': {
                'value': 0,
                'type': 'integer',
            },
            'time_since_cooldown': {
                'value': 0,
                'type': 'integer',
            },
            'time_since_start': {
                'value': 0,
                'type': 'integer',
            },
            'average_active_power': {
                'value': 100,
                'type': 'float',
            },
        }
    },
}

time_step = 1

genset = Genset(genset_params['device'], time_step, real=True)
genset_controller = GensetController(genset_params, time_step)  

## Actions

### Test staying off

In [ ]:
# Test staying off
## If action is "none" and the genset is off, it should stay off
action = {
    'status_change': 'none',
    'power_setpoint': 300
}
safe_action = genset_controller.generate_safe_action(action, reserve_available=True)

# The resulting action status change should be "none" as this is a safe action
assert safe_action['status_change'] == 'none'
# However, as the genset is off, the power setpoint should be 0
assert safe_action['power_setpoint'] == 0

genset.step(action)
genset.print_state()

assert genset.running == False
assert genset_controller.status == 'off'

In [ ]:
## If action is "none" and the genset is off, it should stay off
action = {
    'status_change': 'stop',
    'power_setpoint': 300
}
safe_action = genset_controller.generate_safe_action(action, reserve_available=True)

# The resulting action status change should be "none" as the genset is already off
assert safe_action['status_change'] == 'none'
# However, as the genset is off, the power setpoint should be 0
assert safe_action['power_setpoint'] == 0

genset.step(action)
genset.print_state()

assert genset.running == False
assert genset_controller.status == 'off'

### Test starting process

In [ ]:
# Test start process
timer = 0
## Test first start time step
## If action is "start" and the genset is off, it should start and warm up
action = {
    'status_change': 'start',
    'power_setpoint': 300
}

safe_action = genset_controller.generate_safe_action(action, reserve_available=True)
# The resulting action status change should be "start" as this is a safe action
assert safe_action['status_change'] == 'start'
# The power setpoint should be 100 as this is the warmup power
assert safe_action['power_setpoint'] == 100

genset.step(safe_action)
genset.print_state()

assert genset.running == True
assert genset_controller.status == 'warmup'
assert genset_controller.time_since_warmup == 0
assert genset_controller.time_since_start == 0

In [ ]:
## Test warmup time steps
## In the time steps, the genset should stay in warmup
for i in range(genset_params['controller']['const_params']['warmup_time']['value'] - 1):      # - 1 because there was already one time step when launching the starting action
    action = {
        'status_change': 'none',
        'power_setpoint': 300
    }
    safe_action = genset_controller.generate_safe_action(action, reserve_available=True)

    assert safe_action['status_change'] == 'none'  # The status change should be "none" as the genset is already in warmup
    assert safe_action['power_setpoint'] == 100  # The power setpoint should be 100 as this is the warmup power

    genset.step(safe_action)
    
    print("i: ", i+1, ", genset.running: ", genset.running, ", genset_controller.status: ", genset_controller.status, ", genset_controller.time_since_warmup: ", genset_controller.time_since_warmup, ", genset_controller.time_since_start: ", genset_controller.time_since_start, ", genset.active_power: ", genset.active_power)
    
    assert genset.running == True
    assert genset.active_power == 100  # The active power should be the warmup power
    assert genset_controller.status == 'warmup'
    assert genset_controller.time_since_warmup == i + 1
    assert genset_controller.time_since_start == 0


In [ ]:
## Once the warmup time is over, the genset should be running and not in warmup
action = {
    'status_change': 'none',
    'power_setpoint': 300
}
safe_action = genset_controller.generate_safe_action(action, reserve_available=True)

assert safe_action['status_change'] == 'none'  # The status change should be "none" as the genset is already running
assert safe_action['power_setpoint'] == 300  # The power setpoint should be 300 as we are out of the warmup time
genset.step(safe_action)

print("genset.running: ", genset.running, ", genset_controller.status: ", genset_controller.status, ", genset_controller.time_since_warmup: ", genset_controller.time_since_warmup, ", genset_controller.time_since_start: ", genset_controller.time_since_start, ", genset.active_power: ", genset.active_power)

assert genset.running == True
assert genset_controller.status == 'running'
assert genset.active_power == 300  # The active power should be the one ordered
assert genset_controller.time_since_warmup == 0 # Should be reset
assert genset_controller.time_since_start == 1 # Started to count



In [ ]:
genset_controller.controllable_power

### Test running process

In [ ]:
action = {
    'status_change': 'none',
    'power_setpoint': 300
}
## If action is "none" and the genset is running, it should keep running
safe_action = genset_controller.generate_safe_action(action, reserve_available=True)

assert safe_action['status_change'] == 'none'  # The status change should be "none" as the genset is already running
assert safe_action['power_setpoint'] == 300  # The power setpoint should be 300 as we are out of the warmup time
genset.step(safe_action)

print("genset.running: ", genset.running, ", genset_controller.status: ", genset_controller.status, ", genset_controller.time_since_warmup: ", genset_controller.time_since_warmup, ", genset_controller.time_since_start: ", genset_controller.time_since_start, ", genset.active_power: ", genset.active_power)

assert genset.running == True
assert genset_controller.status == 'running'
assert genset.active_power == 300  # The active power should be the one ordered
assert genset_controller.time_since_warmup == 0 # Should not start counting
assert genset_controller.time_since_start == 2 # Counting should continue

In [ ]:
## If the action is "start" and the genset is running, it should ignore the action and keep running
action = {
    'status_change': 'start',
    'power_setpoint': 300
}
## If action is "start" and the genset is running, it should ignore the "start" and keep running
safe_action = genset_controller.generate_safe_action(action, reserve_available=True)

assert safe_action['status_change'] == 'none'  # The status change should be "none" as the genset is already running
assert safe_action['power_setpoint'] == 300  # The power setpoint should be 300 as commanded
genset.step(safe_action)

print("genset.running: ", genset.running, ", genset_controller.status: ", genset_controller.status, ", genset_controller.time_since_start: ", genset_controller.time_since_start, ", genset.active_power: ", genset.active_power)

assert genset.running == True
assert genset_controller.status == 'running'
assert genset.active_power == 300  # The active power should be the one ordered
assert genset_controller.time_since_start == 3 # Should be incremented by 1

In [ ]:
## If the power setpoint changes, it should follow the new setpoint
action = {
    'status_change': 'none',
    'power_setpoint': 250
}
safe_action = genset_controller.generate_safe_action(action, reserve_available=True)

assert safe_action['status_change'] == 'none'  
assert safe_action['power_setpoint'] == 250  # The power setpoint should be 250 as commanded
genset.step(safe_action)

print("genset.running: ", genset.running, ", genset_controller.status: ", genset_controller.status, ", genset_controller.time_since_start: ", genset_controller.time_since_start, ", genset.active_power: ", genset.active_power)

assert genset.running == True
assert genset_controller.status == 'running'
assert genset.active_power == 250  # The active power should be the one ordered
assert genset_controller.time_since_start == 4 # Should be incremented by 1

In [ ]:
## If the power setpoint is under the minimal load, it should be at the minimal load
action = {
    'status_change': 'none',
    'power_setpoint': 0
}           
safe_action = genset_controller.generate_safe_action(action, reserve_available=True)

assert safe_action['status_change'] == 'none'  
assert safe_action['power_setpoint'] == 120  # The power setpoint should be set at 120 as this is the minimal load
genset.step(safe_action)

print("genset.running: ", genset.running, ", genset_controller.status: ", genset_controller.status, ", genset_controller.time_since_start: ", genset_controller.time_since_start, ", genset.active_power: ", genset.active_power)

assert genset.running == True
assert genset_controller.status == 'running'
assert genset.active_power == 120   # The active power should be the one ordered by the safe action
assert genset_controller.time_since_start == 5 # Should be incremented by 1

In [ ]:
## If the power setpoint is very high and we assume we have access to the reserve, it should be at the prime power times the overload factor
action = {
    'status_change': 'none',
    'power_setpoint': 1000
}

safe_action = genset_controller.generate_safe_action(action, reserve_available=True)

assert safe_action['status_change'] == 'none'  
assert np.abs(safe_action['power_setpoint'] - 440) < 10e-6  # The power setpoint should be set at the maximum available power with reserve (440 kW)
genset.step(safe_action)

print("genset.running: ", genset.running, ", genset_controller.status: ", genset_controller.status, ", genset_controller.time_since_start: ", genset_controller.time_since_start, ", genset.active_power: ", genset.active_power)

assert genset.running == True
assert genset_controller.status == 'running'
assert genset.overload == True     # We are in overload
assert np.abs(genset.active_power - 440) < 10e-6   # The active power should be the one ordered by the safe action


In [ ]:
## Now if we don't have access to the reserve, if the power setpoint is very high, it should be at the prime power only
action = {
    'status_change': 'none',
    'power_setpoint': 1000
}

safe_action = genset_controller.generate_safe_action(action, reserve_available=False) # No access to the reserve

assert safe_action['status_change'] == 'none'  
assert np.abs(safe_action['power_setpoint'] - 400) < 10e-6  # The power setpoint should be set at the maximum available power without reserve (400 kW)
genset.step(safe_action)

print("genset.running: ", genset.running, ", genset_controller.status: ", genset_controller.status, ", genset_controller.time_since_start: ", genset_controller.time_since_start, ", genset.active_power: ", genset.active_power)

assert genset.running == True
assert genset_controller.status == 'running'
assert genset.overload == False     # We are not in overload anymore
assert np.abs(genset.active_power - 400) < 10e-6   # The active power should be the one ordered by the safe action


### Test turning off process

In [ ]:

## Test stopping step
## If action is "stop" and the genset is running but not yet passed the minimum running time, it should not start the cooldown process
action = {
    'status_change': 'stop',
    'power_setpoint': 300
}

safe_action = genset_controller.generate_safe_action(action, reserve_available=True) 

assert safe_action['status_change'] == 'none'  # The status change should be "none" as the genset has not yet passed the minimum running time
assert safe_action['power_setpoint'] == 300  # The power setpoint should be 300 as commanded
genset.step(safe_action)

print("genset.running: ", genset.running, ", genset_controller.status: ", genset_controller.status, ", genset_controller.time_since_start: ", genset_controller.time_since_start, ", genset.active_power: ", genset.active_power)

assert genset.running == True
assert genset_controller.status == 'running'
assert np.abs(genset.active_power - 300) < 10e-6   # The active power should be the one ordered by the safe action

In [ ]:
## Wait for some time until the minimum running time is done
action = {
    'status_change': 'none',
    'power_setpoint': 300
}
for i in range(30):
    safe_action = genset_controller.generate_safe_action(action, reserve_available=True) 
    genset.step(safe_action)
genset.print_state()
print("genset.running: ", genset.running, ", genset_controller.status: ", genset_controller.status, ", genset_controller.time_since_start: ", genset_controller.time_since_start, ", genset.active_power: ", genset.active_power)


In [ ]:
## Stop the genset
action = {
    'status_change': 'stop',
    'power_setpoint': 300
}

safe_action = genset_controller.generate_safe_action(action, reserve_available=True) 

assert safe_action['status_change'] == 'none'  # The status change should be "none" because the genset is getting in cooldown (but it is still running)
assert safe_action['power_setpoint'] == 0  # The power setpoint should be 0 as this is the cooldown power
genset.step(safe_action)

print("genset.running: ", genset.running, ", genset_controller.status: ", genset_controller.status, ", genset_controller.time_since_cooldown: ", genset_controller.time_since_cooldown, ", genset_controller.time_since_start: ", genset_controller.time_since_start, ", genset.active_power: ", genset.active_power)

assert genset.running == True
assert genset_controller.status == 'cooldown'
assert np.abs(genset.active_power) < 10e-6   # The active power should be 0 as this is cooldown

In [ ]:

## Test cooldown time steps
## In the time steps, the genset should stay in cooldown even if we try to start it
for i in range(genset_params['controller']['const_params']['cooldown_time']['value'] - 1):      # - 1 because there was already one time step when launching the starting action, and when we reach the last cooldown we can start it directly
    safe_action = genset_controller.generate_safe_action(action, reserve_available=True) 

    assert safe_action['status_change'] == 'none'  # The status change should be "none" because the genset is still in cooldown
    assert safe_action['power_setpoint'] == 0  # The power setpoint should be 0 as this is the cooldown power
    genset.step(safe_action)

    print("i: ", i, "genset.running: ", genset.running, ", genset_controller.status: ", genset_controller.status, ", genset_controller.time_since_cooldown: ", genset_controller.time_since_cooldown, ", genset_controller.time_since_start: ", genset_controller.time_since_start, ", genset.active_power: ", genset.active_power)

    assert genset.running == True
    assert genset_controller.status == 'cooldown'
    assert np.abs(genset.active_power) < 10e-6   # The active power should be 0 as this is cooldown

In [ ]:
safe_action

In [ ]:
## Once the cooldown time is over, the genset should turn off
action = {
    'status_change': 'none',
    'power_setpoint': 300
}

safe_action = genset_controller.generate_safe_action(action, reserve_available=True) 

assert safe_action['status_change'] == 'stop'  # Now is the time to send the stop signal to the genset
assert safe_action['power_setpoint'] == 0  # The power setpoint should still be 0
genset.step(safe_action)

print("genset.running: ", genset.running, ", genset_controller.status: ", genset_controller.status, ", genset_controller.time_since_cooldown: ", genset_controller.time_since_cooldown, ", genset_controller.time_since_start: ", genset_controller.time_since_start, ", genset.active_power: ", genset.active_power)

assert genset.running == False      # Finally the genset should be off
assert genset_controller.status == 'off'
assert np.abs(genset.active_power) < 10e-6   

In [ ]:
## If I try to stop, it does not change anything
action = {
    'status_change': 'stop',
    'power_setpoint': 300
}

safe_action = genset_controller.generate_safe_action(action, reserve_available=True) 

assert safe_action['status_change'] == 'none'  # Back to "none" as the genset is already off
assert safe_action['power_setpoint'] == 0  # The power setpoint should still be 0
genset.step(safe_action)

print("genset.running: ", genset.running, ", genset_controller.status: ", genset_controller.status, ", genset_controller.time_since_cooldown: ", genset_controller.time_since_cooldown, ", genset_controller.time_since_start: ", genset_controller.time_since_start, ", genset.active_power: ", genset.active_power)

assert genset.running == False      
assert genset_controller.status == 'off'
assert np.abs(genset.active_power) < 10e-6   

In [ ]:
## Same with none, it does not change anything
action = {
    'status_change': 'stop',
    'power_setpoint': 300
}

safe_action = genset_controller.generate_safe_action(action, reserve_available=True) 

assert safe_action['status_change'] == 'none'  # Still "none" as the genset is already off
assert safe_action['power_setpoint'] == 0  # The power setpoint should still be 0
genset.step(safe_action)

print("genset.running: ", genset.running, ", genset_controller.status: ", genset_controller.status, ", genset_controller.time_since_cooldown: ", genset_controller.time_since_cooldown, ", genset_controller.time_since_start: ", genset_controller.time_since_start, ", genset.active_power: ", genset.active_power)

assert genset.running == False      
assert genset_controller.status == 'off'
assert np.abs(genset.active_power) < 10e-6   

In [ ]:

## However we should be able to start the genset (closing the cycle)
action = {
    'status_change': 'start',
    'power_setpoint': 300
}

safe_action = genset_controller.generate_safe_action(action, reserve_available=True) 

assert safe_action['status_change'] == 'start'  # Back to turning it on!
assert safe_action['power_setpoint'] == 100  # The power setpoint should get back to the warmup power
genset.step(safe_action)

print("genset.running: ", genset.running, ", genset_controller.status: ", genset_controller.status, ", genset_controller.time_since_cooldown: ", genset_controller.time_since_cooldown, ", genset_controller.time_since_start: ", genset_controller.time_since_start, ", genset.active_power: ", genset.active_power)

assert genset.running == True      # It's back on!
assert genset_controller.status == 'warmup'
assert np.abs(genset.active_power - 100) < 10e-6   # The active power should be warmup power


## Test the average active power constraint

In [ ]:
genset_params_constrained = copy.deepcopy(genset_params)
genset_params_constrained['controller']['const_params']['average_active_power_constraint']['value'] = True
genset_params_constrained['controller']['init_params']['average_active_power']['value'] = 279.5 # Close to the limit of 280

genset_params_unconstrained = copy.deepcopy(genset_params)
genset_params_unconstrained['controller']['const_params']['average_active_power_constraint']['value'] = False

genset_constrained = Genset(genset_params_constrained['device'], time_step, real=True)
genset_controller_constrained = GensetController(genset_params_constrained, time_step)  

genset_unconstrained = Genset(genset_params_unconstrained['device'], time_step, real=True)
genset_controller_unconstrained = GensetController(genset_params_unconstrained, time_step)  

print("Constrained: ", genset_controller_constrained.average_active_power_constraint, ", Unconstrained: ", genset_controller_unconstrained.average_active_power_constraint)
print("Constrained: ", genset_controller_constrained.average_active_power, ", Unconstrained: ", genset_controller_unconstrained.average_active_power)

In [ ]:
action = {
    'status_change': 'start',
    'power_setpoint': 400               # Ask for prime power
}

active_power_const = []
active_power_unconst = []

average_active_power_const = []
average_active_power_unconst = []


for i in range(50):
    safe_action_constrained = genset_controller_constrained.generate_safe_action(action, reserve_available=True)
    safe_action_unconstrained = genset_controller_unconstrained.generate_safe_action(action, reserve_available=True)

    genset_constrained.step(safe_action_constrained)
    genset_unconstrained.step(safe_action_unconstrained)

    obs_const = {'device_observations': genset_constrained.gather_observations()}
    obs_uncons = {'device_observations': genset_unconstrained.gather_observations()}

    genset_controller_constrained.update_controller_state(obs_const)
    genset_controller_unconstrained.update_controller_state(obs_uncons)

    obs_const_controller = genset_controller_constrained.gather_observations()
    obs_uncons_controller = genset_controller_unconstrained.gather_observations()

    active_power_const.append(obs_const_controller['device_observations']['active_power'])
    active_power_unconst.append(obs_uncons_controller['device_observations']['active_power'])
    average_active_power_const.append(obs_const_controller['controller_state']['average_active_power'])
    average_active_power_unconst.append(obs_uncons_controller['controller_state']['average_active_power'])

import matplotlib.pyplot as plt
plt.figure()
plt.plot(active_power_const, label='Constrained active power')
plt.plot(active_power_unconst, label='Unconstrained active power')
plt.xlabel('Time step')
plt.ylabel('Active power')

plt.figure()
plt.plot(average_active_power_const, label='Constrained average active power')
plt.xlabel('Time step')
plt.ylabel('Cosntrained average active power')

plt.figure()
plt.plot(average_active_power_unconst, label='Unconstrained average active power')
plt.xlabel('Time step')
plt.ylabel('Unconstrained average active power')


assert obs_const_controller['device_observations']['active_power'] == genset_params_constrained['device']['const_params']['prime_power_rating']['value'] * 0.7
assert obs_const_controller['controller_state']['average_active_power'] == genset_params_constrained['device']['const_params']['prime_power_rating']['value'] * 0.7
assert obs_uncons_controller['device_observations']['active_power'] == genset_params_unconstrained['device']['const_params']['prime_power_rating']['value']
assert obs_uncons_controller['controller_state']['average_active_power'] == 0

# Battery

In [ ]:
from battery import Battery
from battery_controller import BatteryController
import numpy as np

In [ ]:
battery_params = {
    "device": {
        "const_params": {
            "P_nom": {"value": 600, "type": "float"},
            "soc_min": {"value": 0, "type": "float"},
            "soc_max": {"value": 1, "type": "float"},
            "I_nom": {"value": 672, "type": "float"},
            "Q_max": {"value": 672, "type": "float"},
            "eta_charge": {"value": 0.95, "type": "float"},
            "R": {"value": 0.039375, "type": "float"},
            "A": {"value": 39.6, "type": "float"},
            "B": {"value": 929.16, "type": "float"},
            "alpha_d": {"value": 5, "type": "float"},
            "beta": {"value": 1, "type": "float"},
            "degradation_cost_type": {"value": "cycle_based", "type": "string"},
            "buffer_size": {"value": 30, "type": "integer"},
            "noise_level": {"value": "none", "type": "string"},
            "noise_range_percentage": {"value": 0.2, "type": "float"},
        },
        "init_params": {
            "soc": {"value": 0.5, "type": "float", "min": 0.1, "max": 0.9},
            "p_grid": {"value": 0, "type": "float"},
            "obs_type": 'params'
        },
    },
    "controller": {
        "const_params": {
            "soc_max_norm": {"value": 0.9, "type": "float"},
            "soc_min_norm": {"value": 0.1, "type": "float"},
            "soc_max_res": {"value": 0.95, "type": "float"},
            "soc_min_res": {"value": 0.05, "type": "float"},
        },
        "init_params": {
            "obs_type": 'params'
        },
    },
}

time_step = 1

battery = Battery(battery_params['device'], time_step, real=True)
battery_controller = BatteryController(battery_params, time_step)



Idle battery (no charge, no discharge) does not change the state

In [ ]:
## Initial state
old_state = battery.gather_observations()
print("Initial state")
print(old_state)

## No power taken - should not change the state
action = {'p_grid': 0}
safe_action = battery_controller.generate_safe_action(action, reserve_available=True)
assert safe_action['p_grid'] == 0  # The power setpoint should be 0 as this is safe
battery.step(safe_action)
new_state = battery.gather_observations()
print("No power taken - should not change the state")
print(new_state)
assert all(np.all(new_state[key] == old_state[key]) for key in new_state)

In [ ]:
old_state = new_state
## Take some power from the battery
action = {'p_grid': 100}
safe_action = battery_controller.generate_safe_action(action, reserve_available=True)
assert safe_action['p_grid'] == 100  # The power setpoint should be 100 as this is safe

print("Action: Battery discharging (p_grid = 100)")
battery.step(safe_action)
new_state = battery.gather_observations()

assert new_state['soc'] < old_state['soc'] # The state of charge should decrease

In [ ]:
prev_state = new_state
## Put back power in the battery
action = {'p_grid': -100}
safe_action = battery_controller.generate_safe_action(action, reserve_available=True)
assert safe_action['p_grid'] == -100  # The power setpoint should be -100 as this is safe

battery.step(safe_action)
new_state = battery.gather_observations()

assert new_state['soc'] > prev_state['soc']
## Because of efficacity losses, the SOC should not be the same
assert new_state['soc'] < old_state['soc']


In [ ]:
# Try to take from the battery more power than it can provide
action = {'p_grid': 1000}
safe_action = battery_controller.generate_safe_action(action, reserve_available=True)
assert safe_action['p_grid'] <= 600  # The power setpoint should be reduced to at least 600 as this is the maximum power available

battery.step(safe_action)
new_state = battery.gather_observations()
print("Actual power taken from the battery: ", new_state['p_grid'])

assert new_state['p_grid'] < action['p_grid']

In [ ]:
# Try to give the battery more power than it can take
action = {'p_grid': -1000}
safe_action = battery_controller.generate_safe_action(action, reserve_available=True)
assert safe_action['p_grid'] >= -600  # The power setpoint should be reduced to at least -600 as this is the maximum power available

battery.step(safe_action)
new_state = battery.gather_observations()
print("Actual power taken from the battery: ", new_state['p_grid'])

assert new_state['p_grid'] > action['p_grid']

In [ ]:
# Charge until max SOC
i = 0
while battery.soc < battery_controller.soc_max_norm:
    action = {'p_grid': -battery.P_nom}
    print("\n --Step {}: ".format(i))
    safe_action = battery_controller.generate_safe_action(action, reserve_available=False)
    battery.step(safe_action)
    
    obs_batt = {'device_observations': battery.gather_observations()}
    
    battery_controller.update_controller_state(obs_batt)
    battery.print_state()
    i+=1

assert np.abs(obs_batt['device_observations']['soc'] - battery_controller.soc_max_norm) < 1e-6

In [ ]:
# Discharge battery until min SOC norm

i = 0
while battery.soc > battery_controller.soc_min_norm + 10e-6:
    action = {'p_grid': battery.P_nom}
    print("\n --Step {}: ".format(i))
    safe_action = battery_controller.generate_safe_action(action, reserve_available=False)
    battery.step(safe_action)
    
    obs_batt = {'device_observations': battery.gather_observations()}
    
    battery_controller.update_controller_state(obs_batt)
    battery.print_state()
    i+=1

assert np.abs(obs_batt['device_observations']['soc'] - battery_controller.soc_min_norm) < 1e-6



In [ ]:
# Test degradation by running the battery for couple of steps.
# Random actions should lead to higher degradation cost than constant actions with final power input/output, and idle actions.

## Init
random_cumul_degradation_cost = 0
random_cumul_degradation_costs = []
random_p_grids = []
random_soc = []

idle_cumul_degradation_cost = 0
idle_cumul_degradation_costs = []
idle_p_grids = []
idle_soc = []

constant_cumul_degradation_cost = 0
constant_cumul_degradation_costs = []
constant_p_grids = []
constant_soc = []

time_steps = 100
seeds = [1, 2, 3, 4, 5, 6, 7, 8, 9, 10]

for seed in seeds:
    print("--Seed {}: ".format(seed))
    np.random.seed(seed)

    random_battery = Battery(battery_params['device'], time_step, real=True)
    random_battery_controller = BatteryController(battery_params, time_step)

    idle_battery = Battery(battery_params['device'], time_step, real=True)
    idle_battery_controller = BatteryController(battery_params, time_step)

    constant_battery= Battery(battery_params['device'], time_step, real=True)
    constant_battery_controller = BatteryController(battery_params, time_step)


    ## Init
    random_cumul_degradation_cost = 0
    random_cumul_degradation_costs = []
    random_p_grids = []
    random_soc = []

    idle_cumul_degradation_cost = 0
    idle_cumul_degradation_costs = []
    idle_p_grids = []
    idle_soc = []

    constant_cumul_degradation_cost = 0
    constant_cumul_degradation_costs = []
    constant_p_grids = []
    constant_soc = []


    for i in range(time_steps):
        # Define actions
        random_action = {'p_grid': np.random.randint(-battery.P_nom, battery.P_nom)}
        idle_action = {'p_grid': 0}

        # Run actions
        random_safe_action = random_battery_controller.generate_safe_action(random_action, reserve_available=False)
        random_battery.step(random_safe_action)

        idle_safe_action = idle_battery_controller.generate_safe_action(idle_action, reserve_available=False)
        idle_battery.step(idle_safe_action)

        # Gather observations
        random_obs_batt = {'device_observations': random_battery.gather_observations()}
        idle_obs_batt = {'device_observations': idle_battery.gather_observations()}

        # Update controller    
        random_battery_controller.update_controller_state(random_obs_batt)
        idle_battery_controller.update_controller_state(idle_obs_batt)

        # Update lists
        random_cumul_degradation_cost += random_obs_batt['device_observations']['degradation_cost']
        random_cumul_degradation_costs.append(random_cumul_degradation_cost)
        random_p_grids.append(random_obs_batt['device_observations']['p_grid'])
        random_soc.append(random_obs_batt['device_observations']['soc'])

        idle_cumul_degradation_cost += idle_obs_batt['device_observations']['degradation_cost']
        idle_cumul_degradation_costs.append(idle_cumul_degradation_cost)
        idle_p_grids.append(idle_obs_batt['device_observations']['p_grid'])
        idle_soc.append(idle_obs_batt['device_observations']['soc'])

    random_total_energy_consumed = np.sum(np.array(random_p_grids))/60      # in kWh
    equivalent_constant_action_power = random_total_energy_consumed*60 / time_steps 

    for i in range(time_steps):

        constant_action = {'p_grid': equivalent_constant_action_power}
        constant_safe_action = constant_battery_controller.generate_safe_action(constant_action, reserve_available=False)
        constant_battery.step(constant_safe_action)
        constant_obs_batt = {'device_observations': constant_battery.gather_observations()}
        constant_battery_controller.update_controller_state(constant_obs_batt)

        constant_cumul_degradation_cost += constant_obs_batt['device_observations']['degradation_cost']
        constant_cumul_degradation_costs.append(constant_cumul_degradation_cost)
        constant_p_grids.append(constant_obs_batt['device_observations']['p_grid'])
        constant_soc.append(constant_obs_batt['device_observations']['soc'])


    assert random_cumul_degradation_cost > constant_cumul_degradation_cost 
    assert constant_cumul_degradation_cost > idle_cumul_degradation_cost
    print("Check!")

# Plot for the last seed
import matplotlib.pyplot as plt
plt.figure()
plt.plot(random_cumul_degradation_costs, label='Random')  
plt.plot(idle_cumul_degradation_costs, label='Idle')
plt.plot(constant_cumul_degradation_costs, label='Constant')
plt.legend()
plt.xlabel('Time step')
plt.ylabel('Cumulative degradation cost')

plt.figure()
plt.plot(random_p_grids, label='Random')
plt.plot(idle_p_grids, label='Idle')
plt.plot(constant_p_grids, label='Constant')
plt.legend()
plt.xlabel('Time step')
plt.ylabel('Power grid')

plt.figure()
plt.plot(random_soc, label='Random')
plt.plot(idle_soc, label='Idle')
plt.plot(constant_soc, label='Constant')
plt.legend()
plt.xlabel('Time step')
plt.ylabel('State of charge')

# Genset group

In [ ]:
from genset_group import GensetGroup
from genset_group_controller import GensetGroupController
import numpy as np

In [ ]:
time_step = 1

genset_params = {
    'device': {
        'const_params': {
            'alpha_g': {
                'value': 0.25,                # in l/kWh
                'type': 'float',
            },
            'beta_g': {
                'value': 10,                  # in l/h
                'type': 'float',
            },
            'prime_power_rating': {
                'value': 400,                 # in kW
                'type': 'float',
            },
            'temp_overload_factor': {
                'value': 1.1,                 # ratio (1.1 for 10% overload)
                'type': 'float',
            },
            'time_step': {
                'value': 1,                   # in minutes
                'type': 'int',
            },
            'noise_level': {
                'value': 'none',              # options: {none, low, medium, high}
                'type': 'string',
            },
            'noise_range_percentage': {
                'value': 0.2,
                'type': 'float',
            },
        },
        'init_params': {
            'running': {
                'value': False,
                'type': 'bool',
            },
            'init_power_setpoint': {
                'value': 300,
                'type': 'float',
            },
        }
    },
    'controller': {
        'const_params': {
            'minimum_run_time': {
                'value': 30,
                'type': 'integer',
            },
            'minimum_load': {
                'value': 120,
                'type': 'float',
            },
            'warmup_power': {
                'value': 100,
                'type': 'float',
            },
            'cooldown_power': {
                'value': 0,
                'type': 'float',
            },
            'warmup_time': {
                'value': 3,
                'type': 'integer',
            },
            'cooldown_time': {
                'value': 6,
                'type': 'integer',
            },
            'average_active_power_constraint': {
                'value': True,
                'type': 'boolean',
            },
            'average_active_power_window': {
                'value': 2880,
                'type': 'integer',
            },
            'average_active_power_limit': {
                'value': 0.7,
                'type': 'float',
            },
        },
        'init_params': {
            'status': {
                'value': 'off',
                'type': 'string',
                'elements': ['off', 'running', 'warmup', 'cooldown'],
            },
            'time_since_warmup': {
                'value': 0,
                'type': 'integer',
            },
            'time_since_cooldown': {
                'value': 0,
                'type': 'integer',
            },
            'time_since_start': {
                'value': 0,
                'type': 'integer',
            },
            'average_active_power': {
                'value': 100,
                'type': 'float',
            },
        }
    },
}


genset_group_params = {
    'device': {
        'const_params': {
            'n_gensets': {
                'value': 3,
                'type': 'integer',
            },
        },
        'init_params': {
            'gensets' :  [genset_params, genset_params, genset_params],
            'obs_type': 'params'
        },
    },
    'controller': {
        'const_params': {
            'priority_order': {
                'value': True,
                'type': 'boolean',
            },
        },
        'init_params': {
            'obs_type': 'params'
        }
    }
}

time_step = 1

genset_group = GensetGroup(genset_group_params['device'], time_step, real=True)
genset_group_controller = GensetGroupController(genset_group_params, time_step)


Initialization. Should be all off, no active power.

In [ ]:
obs = genset_group_controller.gather_observations()
assert obs['device_observations']['genset_group_active_power'] == 0
assert obs['device_observations']['config_lists']['running_gensets_ids'] == []
assert obs['device_observations']['config_lists']['warmup_gensets_ids'] == []
assert obs['device_observations']['config_lists']['cooldown_gensets_ids'] == []
assert obs['device_observations']['config_lists']['off_gensets_ids'] == [0, 1, 2]


Start a generator.
The power setpoint is irrelevant because the generator will only produce the warmup power.

In [ ]:
action = {
    "status_change": "start_next",
    "power_setpoint": 50,
}

safe_action = genset_group_controller.generate_safe_action(action, reserve_available=True)
assert safe_action == {           # The safe action is individual actions for the gensets
    0: {'status_change': 'start', 'power_setpoint': 100},                   # The first genset should start. Power setpoint is 100 because it is in warmup so not controllable.
    1: {'status_change': None, 'power_setpoint': 0},
    2: {'status_change': None, 'power_setpoint': 0},
}

genset_group.step(safe_action, reserve_available=True)
obs_device = {'device_observations': genset_group.gather_observations()}

genset_group_controller.update_controller_state(obs_device)
obs_controller = genset_group_controller.gather_observations()

assert obs_controller['device_observations']['config_lists']['running_gensets_ids'] == []
assert obs_controller['device_observations']['config_lists']['warmup_gensets_ids'] == [0]          # The first genset is in warmup
assert obs_controller['device_observations']['config_lists']['cooldown_gensets_ids'] == []
assert obs_controller['device_observations']['config_lists']['off_gensets_ids'] == [1, 2]
assert obs_controller['device_observations']['genset_group_active_power'] == genset_params['controller']['const_params']['warmup_power']['value']

Run the generators for 2 minutes - Warmup should not be done

In [ ]:
action = {
    "status_change": "none",
    "power_setpoint": 50
}

for i in range(2):
    safe_action = genset_group_controller.generate_safe_action(action, reserve_available=True)
    assert safe_action == {           # The safe action is individual actions for the gensets
        0: {'status_change': None, 'power_setpoint': 100},                   # nothing is happening as it is already on
        1: {'status_change': None, 'power_setpoint': 0},
        2: {'status_change': None, 'power_setpoint': 0},
    }
    genset_group.step(safe_action, reserve_available=True)
    obs_device = {'device_observations': genset_group.gather_observations()}

    genset_group_controller.update_controller_state(obs_device)
    obs_controller = genset_group_controller.gather_observations()

assert obs_controller['device_observations']['config_lists']['running_gensets_ids'] == []
assert obs_controller['device_observations']['config_lists']['warmup_gensets_ids'] == [0]
assert obs_controller['device_observations']['genset_group_active_power'] == genset_params['controller']['const_params']['warmup_power']['value']


In [ ]:
safe_action

Run the generators for one more minute - warmup should be done and the generator should be only running.
Now we are asking for too little power - we should therefore have the minimal for one genset instead of the power we ask for.

In [ ]:
safe_action = genset_group_controller.generate_safe_action(action, reserve_available=True)
assert safe_action == {           # The safe action is individual actions for the gensets
    0: {'status_change': None, 'power_setpoint': 50},                   # Minimum power is given as the power setpoint
    1: {'status_change': None, 'power_setpoint': 0},
    2: {'status_change': None, 'power_setpoint': 0},
}
genset_group.step(safe_action, reserve_available=True)
obs_device = {'device_observations': genset_group.gather_observations()}

genset_group_controller.update_controller_state(obs_device)
obs_controller = genset_group_controller.gather_observations()


assert obs_controller['device_observations']['config_lists']['running_gensets_ids'] == [0]
assert obs_controller['device_observations']['config_lists']['warmup_gensets_ids'] == []

assert obs_controller['device_observations']['genset_group_active_power'] == genset_params['controller']['const_params']['minimum_load']['value']       # Should be the minimum load of the genset because action is lower than that


Setting the power setpoint a bit higher, so that it is in the capabilities of the generator. Now, the group active power should follow the setpoint.

In [ ]:
action = {
    "status_change": "none",
    "power_setpoint": 150
}

safe_action = genset_group_controller.generate_safe_action(action, reserve_available=True)
print("\n Safe action: ", safe_action)

assert safe_action == {           # The safe action is individual actions for the gensets
    0: {'status_change': None, 'power_setpoint': 150},                   # nothing is happening as it is already on
    1: {'status_change': None, 'power_setpoint': 0},
    2: {'status_change': None, 'power_setpoint': 0},
}
genset_group.step(safe_action, reserve_available=True)
obs_device = {'device_observations': genset_group.gather_observations()}

genset_group_controller.update_controller_state(obs_device)
obs_controller = genset_group_controller.gather_observations()


assert obs_controller['device_observations']['config_lists']['running_gensets_ids'] == [0]
assert obs_controller['device_observations']['config_lists']['warmup_gensets_ids'] == []

assert obs_controller['device_observations']['genset_group_active_power'] == 150      

Pushing the power setpoint higher than the limit of a single generator. It should be limited to the max power _including the overload_ of a single generator.

In [ ]:
action = {
    "status_change": "none",
    "power_setpoint": 460
}

safe_action = genset_group_controller.generate_safe_action(action, reserve_available=True)
assert safe_action == {           # The safe action is individual actions for the gensets
    0: {'status_change': None, 'power_setpoint': 440},                   # nothing is happening as it is already on
    1: {'status_change': None, 'power_setpoint': 0},
    2: {'status_change': None, 'power_setpoint': 0},
}
genset_group.step(safe_action, reserve_available=True)
obs_device = {'device_observations': genset_group.gather_observations()}
genset_group_controller.update_controller_state(obs_device)
obs_controller = genset_group_controller.gather_observations()


assert obs_controller['device_observations']['config_lists']['running_gensets_ids'] == [0]
assert obs_controller['device_observations']['config_lists']['warmup_gensets_ids'] == []

assert obs_controller['device_observations']['genset_group_active_power'] == 440   

Starting a new generator. The new power from the warmup allows to reach the group's power setpoint.

In [ ]:
action = {
    "status_change": "start_next",
    "power_setpoint": 460
}

safe_action = genset_group_controller.generate_safe_action(action, reserve_available=True)
assert safe_action == {           # The safe action is individual actions for the gensets
    0: {'status_change': None, 'power_setpoint': 360},                   # nothing is happening as it is already on
    1: {'status_change': 'start', 'power_setpoint': 100},
    2: {'status_change': None, 'power_setpoint': 0},
}
genset_group.step(safe_action, reserve_available=True)
obs_device = {'device_observations': genset_group.gather_observations()}
genset_group_controller.update_controller_state(obs_device)
obs_controller = genset_group_controller.gather_observations()


assert obs_controller['device_observations']['config_lists']['running_gensets_ids'] == [0]
assert obs_controller['device_observations']['config_lists']['warmup_gensets_ids'] == [1]
assert obs_controller['device_observations']['gensets'][0]['device_observations']['active_power'] == 360
assert obs_controller['device_observations']['gensets'][1]['device_observations']['active_power'] == 100
assert obs_controller['device_observations']['genset_group_active_power'] == 460   


After 2 more time steps, should be the same thing, as the warmup is not done. The controllable generator produces the biggest amount.

In [ ]:
action = {
    "status_change": 'none',
    "power_setpoint": 460
}

for i in range(2):
    safe_action = genset_group_controller.generate_safe_action(action, reserve_available=True)
    assert safe_action == {           # The safe action is individual actions for the gensets
        0: {'status_change': None, 'power_setpoint': 360},                   # nothing is happening as it is already on
        1: {'status_change': None, 'power_setpoint': 100},
        2: {'status_change': None, 'power_setpoint': 0},
    }
    genset_group.step(safe_action, reserve_available=True)
    obs_device = {'device_observations': genset_group.gather_observations()}
    genset_group_controller.update_controller_state(obs_device)
    obs_controller = genset_group_controller.gather_observations()


    assert obs_controller['device_observations']['config_lists']['running_gensets_ids'] == [0]
    assert obs_controller['device_observations']['config_lists']['warmup_gensets_ids'] == [1]
    assert obs_controller['device_observations']['gensets'][0]['device_observations']['active_power'] == 360
    assert obs_controller['device_observations']['gensets'][1]['device_observations']['active_power'] == 100
    assert obs_controller['device_observations']['genset_group_active_power'] == 460   


In [ ]:
safe_action

One more time step, we are done with the warmup. Both generators produce the same amount (as they are exactly similar)

In [ ]:
safe_action = genset_group_controller.generate_safe_action(action, reserve_available=True)
assert safe_action == {           # The safe action is individual actions for the gensets
    0: {'status_change': None, 'power_setpoint': 230},                   # nothing is happening as it is already on
    1: {'status_change': None, 'power_setpoint': 230},
    2: {'status_change': None, 'power_setpoint': 0},
}
genset_group.step(safe_action, reserve_available=True)
obs_device = {'device_observations': genset_group.gather_observations()}
genset_group_controller.update_controller_state(obs_device)
obs_controller = genset_group_controller.gather_observations()


assert obs_controller['device_observations']['config_lists']['running_gensets_ids'] == [0,1]
assert obs_controller['device_observations']['config_lists']['warmup_gensets_ids'] == []
assert obs_controller['device_observations']['gensets'][0]['device_observations']['active_power'] == 230
assert obs_controller['device_observations']['gensets'][1]['device_observations']['active_power'] == 230
assert obs_controller['device_observations']['genset_group_active_power'] == 460   



Passing 10 time steps, the situation should be the same

In [ ]:
for i in range(10):
    safe_action = genset_group_controller.generate_safe_action(action, reserve_available=True)
    assert safe_action == {           # The safe action is individual actions for the gensets
        0: {'status_change': None, 'power_setpoint': 230},                   # nothing is happening as it is already on
        1: {'status_change': None, 'power_setpoint': 230},
        2: {'status_change': None, 'power_setpoint': 0},
    }
    genset_group.step(safe_action, reserve_available=True)
    obs_device = {'device_observations': genset_group.gather_observations()}
    genset_group_controller.update_controller_state(obs_device)
    obs_controller = genset_group_controller.gather_observations()


    assert obs_controller['device_observations']['config_lists']['running_gensets_ids'] == [0,1]
    assert obs_controller['device_observations']['config_lists']['warmup_gensets_ids'] == []
    assert obs_controller['device_observations']['gensets'][0]['device_observations']['active_power'] == 230
    assert obs_controller['device_observations']['gensets'][1]['device_observations']['active_power'] == 230
    assert obs_controller['device_observations']['genset_group_active_power'] == 460   

It's not yet been 30 minutes so we should not be able to turn off a generator. The "stop_last" action is ignored.

In [ ]:
action = {
    "status_change": "stop_last",
    "power_setpoint": 460
}

safe_action = genset_group_controller.generate_safe_action(action, reserve_available=True)
assert safe_action == {           # The safe action is individual actions for the gensets
    0: {'status_change': None, 'power_setpoint': 230},                   # nothing is happening as it is already on
    1: {'status_change': None, 'power_setpoint': 230},
    2: {'status_change': None, 'power_setpoint': 0},
}

genset_group.step(safe_action, reserve_available=True)
obs_device = {'device_observations': genset_group.gather_observations()}
genset_group_controller.update_controller_state(obs_device)
obs_controller = genset_group_controller.gather_observations()


assert obs_controller['device_observations']['config_lists']['running_gensets_ids'] == [0,1]
assert obs_controller['device_observations']['config_lists']['warmup_gensets_ids'] == []
assert obs_controller['device_observations']['gensets'][0]['device_observations']['active_power'] == 230
assert obs_controller['device_observations']['gensets'][1]['device_observations']['active_power'] == 230
assert obs_controller['device_observations']['genset_group_active_power'] == 460   


Let's run 13 minutes more. Now generator 0 should have passed the 30 minutes mark, but not generator 1. We still cannot stop any generator, for priority reasons. 

In [ ]:
action = {
    "status_change": "none",
    "power_setpoint": 460
}

for i in range(13):
    safe_action = genset_group_controller.generate_safe_action(action, reserve_available=True)
    assert safe_action == {           # The safe action is individual actions for the gensets
        0: {'status_change': None, 'power_setpoint': 230},                   # nothing is happening as it is already on
        1: {'status_change': None, 'power_setpoint': 230},
        2: {'status_change': None, 'power_setpoint': 0},
    }

    genset_group.step(safe_action, reserve_available=True)
    obs_device = {'device_observations': genset_group.gather_observations()}
    genset_group_controller.update_controller_state(obs_device)
    obs_controller = genset_group_controller.gather_observations()

print("Time since start for genset 0: ", obs_controller['device_observations']['gensets'][0]['controller_state']['time_since_start'])
print("Time since start for genset 1: ", obs_controller['device_observations']['gensets'][1]['controller_state']['time_since_start'])

action = {
    "status_change": "stop_last",
    "power_setpoint": 460
}

safe_action = genset_group_controller.generate_safe_action(action, reserve_available=True)
assert safe_action == {           # The safe action is individual actions for the gensets
    0: {'status_change': None, 'power_setpoint': 230},                   # nothing is happening as it is already on
    1: {'status_change': None, 'power_setpoint': 230},
    2: {'status_change': None, 'power_setpoint': 0},
}

genset_group.step(safe_action, reserve_available=True)
obs_device = {'device_observations': genset_group.gather_observations()}
genset_group_controller.update_controller_state(obs_device)
obs_controller = genset_group_controller.gather_observations()

assert obs_controller['device_observations']['config_lists']['running_gensets_ids'] == [0,1]
assert obs_controller['device_observations']['config_lists']['warmup_gensets_ids'] == []
assert obs_controller['device_observations']['gensets'][0]['device_observations']['active_power'] == 230
assert obs_controller['device_observations']['gensets'][1]['device_observations']['active_power'] == 230
assert obs_controller['device_observations']['genset_group_active_power'] == 460   

Let's run 5 minutes more. Now, we should be able to stop generator 1.

In [ ]:
action = {
    "status_change": "none",
    "power_setpoint": 460
}

for i in range(5):
    safe_action = genset_group_controller.generate_safe_action(action, reserve_available=True)
    assert safe_action == {           # The safe action is individual actions for the gensets
        0: {'status_change': None, 'power_setpoint': 230},                   # nothing is happening as it is already on
        1: {'status_change': None, 'power_setpoint': 230},
        2: {'status_change': None, 'power_setpoint': 0},
    }

    genset_group.step(safe_action, reserve_available=True)
    obs_device = {'device_observations': genset_group.gather_observations()}
    genset_group_controller.update_controller_state(obs_device)
    obs_controller = genset_group_controller.gather_observations()


assert obs_controller['device_observations']['config_lists']['running_gensets_ids'] == [0,1]
assert obs_controller['device_observations']['config_lists']['warmup_gensets_ids'] == []
assert obs_controller['device_observations']['gensets'][0]['device_observations']['active_power'] == 230
assert obs_controller['device_observations']['gensets'][1]['device_observations']['active_power'] == 230
assert obs_controller['device_observations']['genset_group_active_power'] == 460   


print("Time since start for genset 0: ", obs_controller['device_observations']['gensets'][0]['controller_state']['time_since_start'])
print("Time since start for genset 1: ", obs_controller['device_observations']['gensets'][1]['controller_state']['time_since_start'])

## Let's try to stop the genset

action = {
    "status_change": "stop_last",
    "power_setpoint": 460
}

safe_action = genset_group_controller.generate_safe_action(action, reserve_available=True)
assert safe_action == {           # The safe action is individual actions for the gensets
    0: {'status_change': None, 'power_setpoint': 440},                   # Pushed to the maximum (overload)
    1: {'status_change': 'stop', 'power_setpoint': 0},                  # The second genset should stop. Power setpoint is 0 because it is in cooldown.
    2: {'status_change': None, 'power_setpoint': 0},
}

genset_group.step(safe_action, reserve_available=True)
obs_device = {'device_observations': genset_group.gather_observations()}
genset_group_controller.update_controller_state(obs_device)
obs_controller = genset_group_controller.gather_observations()


assert obs_controller['device_observations']['config_lists']['running_gensets_ids'] == [0]
assert obs_controller['device_observations']['config_lists']['warmup_gensets_ids'] == []
assert obs_controller['device_observations']['config_lists']['cooldown_gensets_ids'] == [1]
assert obs_controller['device_observations']['config_lists']['off_gensets_ids'] == [2]
assert obs_controller['device_observations']['gensets'][0]['device_observations']['active_power'] == 440
assert obs_controller['device_observations']['gensets'][1]['device_observations']['active_power'] == 0
assert obs_controller['device_observations']['genset_group_active_power'] == 440   




If we try to turn on a generator again, we cannot because we have to wait for the cooldown to be done. Nothing happens.

In [ ]:
action = {
    "status_change": "start_next",
    "power_setpoint": 400
}


safe_action = genset_group_controller.generate_safe_action(action, reserve_available=True)
assert safe_action == {           # The safe action is individual actions for the gensets
    0: {'status_change': None, 'power_setpoint': 400},                   # Pushed to 400
    1: {'status_change': None, 'power_setpoint': 0},                
    2: {'status_change': None, 'power_setpoint': 0},
}

genset_group.step(safe_action, reserve_available=True)
obs_device = {'device_observations': genset_group.gather_observations()}
genset_group_controller.update_controller_state(obs_device)
obs_controller = genset_group_controller.gather_observations()


assert obs_controller['device_observations']['config_lists']['running_gensets_ids'] == [0]
assert obs_controller['device_observations']['config_lists']['warmup_gensets_ids'] == []
assert obs_controller['device_observations']['config_lists']['cooldown_gensets_ids'] == [1]
assert obs_controller['device_observations']['config_lists']['off_gensets_ids'] == [2]
assert obs_controller['device_observations']['gensets'][0]['device_observations']['active_power'] == 400
assert obs_controller['device_observations']['gensets'][1]['device_observations']['active_power'] == 0
assert obs_controller['device_observations']['genset_group_active_power'] == 400   

However, we should also be able to turn off the other generator.

In [ ]:
action = {
    "status_change": "stop_last",
    "power_setpoint": 450
}


safe_action = genset_group_controller.generate_safe_action(action, reserve_available=True)
assert safe_action == {           # The safe action is individual actions for the gensets
    0: {'status_change': 'stop', 'power_setpoint': 0},                   # Turning it off
    1: {'status_change': None, 'power_setpoint': 0},                  
    2: {'status_change': None, 'power_setpoint': 0},
}

genset_group.step(safe_action, reserve_available=True)
obs_device = {'device_observations': genset_group.gather_observations()}
genset_group_controller.update_controller_state(obs_device)
obs_controller = genset_group_controller.gather_observations()


assert obs_controller['device_observations']['config_lists']['running_gensets_ids'] == []
assert obs_controller['device_observations']['config_lists']['warmup_gensets_ids'] == []
assert obs_controller['device_observations']['config_lists']['cooldown_gensets_ids'] == [0,1]
assert obs_controller['device_observations']['config_lists']['off_gensets_ids'] == [2]
assert obs_controller['device_observations']['gensets'][0]['device_observations']['active_power'] == 0
assert obs_controller['device_observations']['gensets'][1]['device_observations']['active_power'] == 0
assert obs_controller['device_observations']['genset_group_active_power'] == 0   

Now we can must wait generator 0 to be completely off to start one again - even if generator 1 is done, it would not work with the priority rules. start_next is ingored. 

In [ ]:
action = {
    "status_change": "start_next",
    "power_setpoint": 400
}

for i in range(5):
    safe_action = genset_group_controller.generate_safe_action(action, reserve_available=True)
    assert safe_action == {           # The safe action is individual actions for the gensets
        0: {'status_change': None, 'power_setpoint': 0},                   # Off
        1: {'status_change': None, 'power_setpoint': 0},                  
        2: {'status_change': None, 'power_setpoint': 0},
    }

    genset_group.step(safe_action, reserve_available=True)
    obs_device = {'device_observations': genset_group.gather_observations()}
    genset_group_controller.update_controller_state(obs_device)
    obs_controller = genset_group_controller.gather_observations()


    assert obs_controller['device_observations']['config_lists']['running_gensets_ids'] == []
    assert obs_controller['device_observations']['config_lists']['warmup_gensets_ids'] == []
    assert obs_controller['device_observations']['gensets'][0]['controller_state']['status'] == 'cooldown'
    assert obs_controller['device_observations']['gensets'][1]['controller_state']['status'] in ['cooldown', 'off']
    assert obs_controller['device_observations']['gensets'][0]['device_observations']['active_power'] == 0
    assert obs_controller['device_observations']['gensets'][1]['device_observations']['active_power'] == 0
    assert obs_controller['device_observations']['genset_group_active_power'] == 0  


# Check that 1 is off and 0 is in cooldown
assert obs_controller['device_observations']['config_lists']['running_gensets_ids'] == []
assert obs_controller['device_observations']['config_lists']['warmup_gensets_ids'] == []
assert obs_controller['device_observations']['gensets'][0]['controller_state']['status'] == 'cooldown'
assert obs_controller['device_observations']['gensets'][1]['controller_state']['status'] == 'off'
assert obs_controller['device_observations']['gensets'][0]['device_observations']['active_power'] == 0
assert obs_controller['device_observations']['gensets'][1]['device_observations']['active_power'] == 0
assert obs_controller['device_observations']['genset_group_active_power'] == 0  

Next time step would start it though

In [ ]:
action = {
    "status_change": "start_next",
    "power_setpoint": 400
}

safe_action = genset_group_controller.generate_safe_action(action, reserve_available=True)
assert safe_action == {           # The safe action is individual actions for the gensets
    0: {'status_change': 'start', 'power_setpoint': 100},                   # Warmup power
    1: {'status_change': None, 'power_setpoint': 0},                  
    2: {'status_change': None, 'power_setpoint': 0},
}

genset_group.step(safe_action, reserve_available=True)
obs_device = {'device_observations': genset_group.gather_observations()}
genset_group_controller.update_controller_state(obs_device)
obs_controller = genset_group_controller.gather_observations()


assert obs_controller['device_observations']['config_lists']['running_gensets_ids'] == []
assert obs_controller['device_observations']['config_lists']['warmup_gensets_ids'] == [0]
assert obs_controller['device_observations']['gensets'][0]['controller_state']['status'] == 'warmup'
assert obs_controller['device_observations']['gensets'][1]['controller_state']['status'] == 'off'
assert obs_controller['device_observations']['gensets'][0]['device_observations']['active_power'] == 100
assert obs_controller['device_observations']['gensets'][1]['device_observations']['active_power'] == 0
assert obs_controller['device_observations']['genset_group_active_power'] == 100  



While it is starting, in the warmup phase, we cannot stop it. A stop signal will do nothing.

In [ ]:
action = {
    "status_change": "stop_last",
    "power_setpoint": 400
}

safe_action = genset_group_controller.generate_safe_action(action, reserve_available=True)
assert safe_action == {           # The safe action is individual actions for the gensets
    0: {'status_change': None, 'power_setpoint': 100},                   # Still warmup power
    1: {'status_change': None, 'power_setpoint': 0},                  
    2: {'status_change': None, 'power_setpoint': 0},
}

genset_group.step(safe_action, reserve_available=True)
obs_device = {'device_observations': genset_group.gather_observations()}
genset_group_controller.update_controller_state(obs_device)
obs_controller = genset_group_controller.gather_observations()


assert obs_controller['device_observations']['config_lists']['running_gensets_ids'] == []
assert obs_controller['device_observations']['config_lists']['warmup_gensets_ids'] == [0]
assert obs_controller['device_observations']['gensets'][0]['controller_state']['status'] == 'warmup'
assert obs_controller['device_observations']['gensets'][1]['controller_state']['status'] == 'off'
assert obs_controller['device_observations']['gensets'][0]['device_observations']['active_power'] == 100
assert obs_controller['device_observations']['gensets'][1]['device_observations']['active_power'] == 0
assert obs_controller['device_observations']['genset_group_active_power'] == 100  

However we can start all the generators now.

In [ ]:
action = {
    "status_change": "start_next",
    "power_setpoint": 400
}

# Turn on genset 1

safe_action = genset_group_controller.generate_safe_action(action, reserve_available=True)
assert safe_action == {           # The safe action is individual actions for the gensets
    0: {'status_change': None, 'power_setpoint': 100},                   # Still warmup power
    1: {'status_change': 'start', 'power_setpoint': 100},                # This one starts
    2: {'status_change': None, 'power_setpoint': 0},
}

genset_group.step(safe_action, reserve_available=True)
obs_device = {'device_observations': genset_group.gather_observations()}
genset_group_controller.update_controller_state(obs_device)
obs_controller = genset_group_controller.gather_observations()


assert obs_controller['device_observations']['config_lists']['running_gensets_ids'] == []
assert obs_controller['device_observations']['config_lists']['warmup_gensets_ids'] == [0, 1]
assert obs_controller['device_observations']['gensets'][0]['controller_state']['status'] == 'warmup'
assert obs_controller['device_observations']['gensets'][1]['controller_state']['status'] == 'warmup'
assert obs_controller['device_observations']['gensets'][0]['device_observations']['active_power'] == 100
assert obs_controller['device_observations']['gensets'][1]['device_observations']['active_power'] == 100
assert obs_controller['device_observations']['genset_group_active_power'] == 200  


# Turn on genset 2. Also, genset 0 is done warming up.
safe_action = genset_group_controller.generate_safe_action(action, reserve_available=True)
assert safe_action == {           # The safe action is individual actions for the gensets
    0: {'status_change': None, 'power_setpoint': 200},                   # Running, producing to meet the setpoint
    1: {'status_change': None, 'power_setpoint': 100},                # This one is already in warmup
    2: {'status_change': 'start', 'power_setpoint': 100},               # Time for this one to warmup
}

genset_group.step(safe_action, reserve_available=True)
obs_device = {'device_observations': genset_group.gather_observations()}
genset_group_controller.update_controller_state(obs_device)
obs_controller = genset_group_controller.gather_observations()


assert obs_controller['device_observations']['config_lists']['running_gensets_ids'] == [0]
assert obs_controller['device_observations']['config_lists']['warmup_gensets_ids'] == [1, 2]
assert obs_controller['device_observations']['gensets'][0]['controller_state']['status'] == 'running'
assert obs_controller['device_observations']['gensets'][1]['controller_state']['status'] == 'warmup'
assert obs_controller['device_observations']['gensets'][2]['controller_state']['status'] == 'warmup'
assert obs_controller['device_observations']['gensets'][0]['device_observations']['active_power'] == 200
assert obs_controller['device_observations']['gensets'][1]['device_observations']['active_power'] == 100
assert obs_controller['device_observations']['gensets'][2]['device_observations']['active_power'] == 100
assert obs_controller['device_observations']['genset_group_active_power'] == 400  


If we do it once more, it does not do anything of course because there are only 3 generators.

In [ ]:
action = {
    "status_change": "start_next",
    "power_setpoint": 400
}

safe_action = genset_group_controller.generate_safe_action(action, reserve_available=True)
assert safe_action == {           # The safe action is individual actions for the gensets
    0: {'status_change': None, 'power_setpoint': 200},                   # Running, producing to meet the setpoint
    1: {'status_change': None, 'power_setpoint': 100},                # This one is already in warmup
    2: {'status_change': None, 'power_setpoint': 100},               # This one is already in warmup
}

genset_group.step(safe_action, reserve_available=True)
obs_device = {'device_observations': genset_group.gather_observations()}
genset_group_controller.update_controller_state(obs_device)
obs_controller = genset_group_controller.gather_observations()


assert obs_controller['device_observations']['config_lists']['running_gensets_ids'] == [0]
assert obs_controller['device_observations']['config_lists']['warmup_gensets_ids'] == [1, 2]
assert obs_controller['device_observations']['gensets'][0]['controller_state']['status'] == 'running'
assert obs_controller['device_observations']['gensets'][1]['controller_state']['status'] == 'warmup'
assert obs_controller['device_observations']['gensets'][2]['controller_state']['status'] == 'warmup'
assert obs_controller['device_observations']['gensets'][0]['device_observations']['active_power'] == 200
assert obs_controller['device_observations']['gensets'][1]['device_observations']['active_power'] == 100
assert obs_controller['device_observations']['gensets'][2]['device_observations']['active_power'] == 100
assert obs_controller['device_observations']['genset_group_active_power'] == 400  

Let's run for a bit... They should all get out of warmup and split the load equally.

In [ ]:
action = {
    "status_change": "none",
    "power_setpoint": 400
}

for i in range(10):
    safe_action = genset_group_controller.generate_safe_action(action, reserve_available=True)

    genset_group.step(safe_action, reserve_available=True)
    obs_device = {'device_observations': genset_group.gather_observations()}
    genset_group_controller.update_controller_state(obs_device)
    obs_controller = genset_group_controller.gather_observations()
    assert np.round(obs_controller['device_observations']['genset_group_active_power'], 1) == 400  

assert obs_controller['device_observations']['config_lists']['running_gensets_ids'] == [0, 1, 2]
assert obs_controller['device_observations']['config_lists']['warmup_gensets_ids'] == []
assert obs_controller['device_observations']['gensets'][0]['controller_state']['status'] == 'running'
assert obs_controller['device_observations']['gensets'][1]['controller_state']['status'] == 'running'
assert obs_controller['device_observations']['gensets'][2]['controller_state']['status'] == 'running'
assert np.round(obs_controller['device_observations']['gensets'][0]['device_observations']['active_power'], 1) == np.round(400/3, 1)
assert np.round(obs_controller['device_observations']['gensets'][1]['device_observations']['active_power'], 1) == np.round(400/3, 1)
assert np.round(obs_controller['device_observations']['gensets'][2]['device_observations']['active_power'], 1) == np.round(400/3, 1)
assert np.round(obs_controller['device_observations']['genset_group_active_power'], 1) == 400  


Let's try to produce just a tiny bit of power: we are limited to the lower limit

In [ ]:
action = {
    "status_change": "none",
    "power_setpoint": 200
}

safe_action = genset_group_controller.generate_safe_action(action, reserve_available=True)

assert safe_action == {           # The safe action is individual actions for the gensets
    0: {'status_change': None, 'power_setpoint': np.round(200/3, 2)},                   # The safe action does not take into account the minimum load of the genset
    1: {'status_change': None, 'power_setpoint': np.round(200/3, 2)},                
    2: {'status_change': None, 'power_setpoint': np.round(200/3, 2)},               
}


genset_group.step(safe_action, reserve_available=True)
obs_device = {'device_observations': genset_group.gather_observations()}
genset_group_controller.update_controller_state(obs_device)
obs_controller = genset_group_controller.gather_observations()

assert obs_controller['device_observations']['config_lists']['running_gensets_ids'] == [0, 1, 2]
assert obs_controller['device_observations']['config_lists']['warmup_gensets_ids'] == []
assert obs_controller['device_observations']['gensets'][0]['controller_state']['status'] == 'running'
assert obs_controller['device_observations']['gensets'][1]['controller_state']['status'] == 'running'
assert obs_controller['device_observations']['gensets'][2]['controller_state']['status'] == 'running'
assert obs_controller['device_observations']['gensets'][0]['device_observations']['active_power'] == 120        # Yet the genset cannot provide less than their minimum load of 120
assert obs_controller['device_observations']['gensets'][1]['device_observations']['active_power'] == 120
assert obs_controller['device_observations']['gensets'][2]['device_observations']['active_power'] == 120
assert obs_controller['device_observations']['genset_group_active_power'] == 360  


In [ ]:
safe_action

Now, let's ask for too much

In [ ]:
action = {
    "status_change": "none",
    "power_setpoint": 2000
}

safe_action = genset_group_controller.generate_safe_action(action, reserve_available=True)

assert safe_action == {           # The safe action is individual actions for the gensets
    0: {'status_change': None, 'power_setpoint': 440},                   # The safe action cannot give more than the maximum available power of the gensets
    1: {'status_change': None, 'power_setpoint': 440},                
    2: {'status_change': None, 'power_setpoint': 440},               
}


genset_group.step(safe_action, reserve_available=True)
obs_device = {'device_observations': genset_group.gather_observations()}
genset_group_controller.update_controller_state(obs_device)
obs_controller = genset_group_controller.gather_observations()

assert obs_controller['device_observations']['config_lists']['running_gensets_ids'] == [0, 1, 2]
assert obs_controller['device_observations']['config_lists']['warmup_gensets_ids'] == []
assert obs_controller['device_observations']['gensets'][0]['controller_state']['status'] == 'running'
assert obs_controller['device_observations']['gensets'][1]['controller_state']['status'] == 'running'
assert obs_controller['device_observations']['gensets'][2]['controller_state']['status'] == 'running'
assert obs_controller['device_observations']['gensets'][0]['device_observations']['active_power'] == 440        # Yet the genset cannot provide less than their minimum load of 120
assert obs_controller['device_observations']['gensets'][1]['device_observations']['active_power'] == 440
assert obs_controller['device_observations']['gensets'][2]['device_observations']['active_power'] == 440
assert obs_controller['device_observations']['genset_group_active_power'] == 440*3  




Let's run for a while... to raise the average active power. The limit on the average active power over the last 48 hours is 0.7*prime power, so here, 280 kW. 
After 30 hours (1800 min), not yet...

In [ ]:
import matplotlib.pyplot as plt

action = {
    "status_change": "none",
    "power_setpoint": 2000
}

genset_group_active_power = []
genset_1_power = []
genset_1_avgpower = []
genset_2_power = []
genset_2_avgpower = []
genset_3_power = []
genset_3_avgpower = []


for i in range(1600):
    safe_action = genset_group_controller.generate_safe_action(action, reserve_available=True)
    genset_group.step(safe_action, reserve_available=True)
    obs_device = {'device_observations': genset_group.gather_observations()}
    genset_group_controller.update_controller_state(obs_device)
    obs_controller = genset_group_controller.gather_observations()



    genset_group_active_power.append(obs_controller['device_observations']['genset_group_active_power'])
    genset_1_power.append(obs_controller['device_observations']['gensets'][0]['device_observations']['active_power'])
    genset_1_avgpower.append(obs_controller['device_observations']['gensets'][0]['controller_state']['average_active_power'])
    genset_2_power.append(obs_controller['device_observations']['gensets'][1]['device_observations']['active_power'])
    genset_2_avgpower.append(obs_controller['device_observations']['gensets'][1]['controller_state']['average_active_power'])
    genset_3_power.append(obs_controller['device_observations']['gensets'][2]['device_observations']['active_power'])
    genset_3_avgpower.append(obs_controller['device_observations']['gensets'][2]['controller_state']['average_active_power'])


plt.plot(genset_group_active_power, label="Genset group active power")
plt.plot(genset_1_power, label="Genset 1 active power")
plt.plot(genset_1_avgpower, label="Genset 1 average power", linestyle='dashed')
plt.plot(genset_2_power, label="Genset 2 active power")
plt.plot(genset_2_avgpower, label="Genset 2 average power", linestyle='dashed')
plt.plot(genset_3_power, label="Genset 3 active power")
plt.plot(genset_3_avgpower, label="Genset 3 average power", linestyle='dashed')
plt.legend(["Genset group active power", "Genset 1 active power", "Genset 1 average power", "Genset 2 active power", "Genset 2 average power", "Genset 3 active power", "Genset 3 average power"])


assert obs_controller['device_observations']['config_lists']['running_gensets_ids'] == [0, 1, 2]
assert obs_controller['device_observations']['config_lists']['warmup_gensets_ids'] == []
assert obs_controller['device_observations']['gensets'][0]['controller_state']['status'] == 'running'
assert obs_controller['device_observations']['gensets'][1]['controller_state']['status'] == 'running'
assert obs_controller['device_observations']['gensets'][2]['controller_state']['status'] == 'running'
assert obs_controller['device_observations']['gensets'][0]['device_observations']['active_power'] == 440        # Yet the genset cannot provide less than their minimum load of 120
assert obs_controller['device_observations']['gensets'][1]['device_observations']['active_power'] == 440
assert obs_controller['device_observations']['gensets'][2]['device_observations']['active_power'] == 440
assert obs_controller['device_observations']['genset_group_active_power'] == 440*3  


After a bit more, here we are, limited to 280 kW for each genset.

In [ ]:
action = {
    "status_change": "none",
    "power_setpoint": 2000
}

for i in range(1200):
    safe_action = genset_group_controller.generate_safe_action(action, reserve_available=True)
    genset_group.step(safe_action, reserve_available=True)
    obs_device = {'device_observations': genset_group.gather_observations()}
    genset_group_controller.update_controller_state(obs_device)
    obs_controller = genset_group_controller.gather_observations()

    genset_group_active_power.append(obs_controller['device_observations']['genset_group_active_power'])
    genset_1_power.append(obs_controller['device_observations']['gensets'][0]['device_observations']['active_power'])
    genset_1_avgpower.append(obs_controller['device_observations']['gensets'][0]['controller_state']['average_active_power'])
    genset_2_power.append(obs_controller['device_observations']['gensets'][1]['device_observations']['active_power'])
    genset_2_avgpower.append(obs_controller['device_observations']['gensets'][1]['controller_state']['average_active_power'])
    genset_3_power.append(obs_controller['device_observations']['gensets'][2]['device_observations']['active_power'])
    genset_3_avgpower.append(obs_controller['device_observations']['gensets'][2]['controller_state']['average_active_power'])

plt.plot(genset_group_active_power, label="Genset group active power")
plt.plot(genset_1_power, label="Genset 1 active power")
plt.plot(genset_1_avgpower, label="Genset 1 average power", linestyle='dashed')
plt.plot(genset_2_power, label="Genset 2 active power")
plt.plot(genset_2_avgpower, label="Genset 2 average power", linestyle='dashed')
plt.plot(genset_3_power, label="Genset 3 active power")
plt.plot(genset_3_avgpower, label="Genset 3 average power", linestyle='dashed')
plt.legend(["Genset group active power", "Genset 1 active power", "Genset 1 average power", "Genset 2 active power", "Genset 2 average power", "Genset 3 active power", "Genset 3 average power"])



assert obs_controller['device_observations']['gensets'][0]['device_observations']['active_power'] == 280        # Yet the genset cannot provide less than their minimum load of 120
assert obs_controller['device_observations']['gensets'][1]['device_observations']['active_power'] == 280
assert obs_controller['device_observations']['gensets'][2]['device_observations']['active_power'] == 280
assert obs_controller['device_observations']['genset_group_active_power'] == 280*3  



Let's now test the case where the average power is different for the gensets.


In [ ]:
import copy
time_step = 1


limit_genset_params = copy.deepcopy(genset_params)
limit_genset_params['controller']['init_params']['status']['value'] = 'running'       # Start running  
limit_genset_params['device']['init_params']['running']['value'] = True  
limit_genset_params['controller']['init_params']['average_active_power']['value'] = 275         # Start high

no_limit_genset_params = copy.deepcopy(limit_genset_params)
no_limit_genset_params['controller']['init_params']['average_active_power']['value'] = 0         # Start low

new_genset_group_params = copy.deepcopy(genset_group_params)
new_genset_group_params['device']['init_params']['gensets'] = [limit_genset_params, no_limit_genset_params, no_limit_genset_params]     # We start with one that is limited and two that are not


genset_group = GensetGroup(new_genset_group_params['device'], time_step, real=True)
genset_group_controller = GensetGroupController(new_genset_group_params, time_step)

We expect the three gensets to be limited by the limit of the first one, to ensure the respect of the constraint stating that the same ratio of prime power must be sent to all gensets.

In [ ]:
action = {
    "status_change": "none",
    "power_setpoint": 2000
}


genset_group_active_power = []
genset_1_power = []
genset_1_avgpower = []
genset_2_power = []
genset_2_avgpower = []
genset_3_power = []
genset_3_avgpower = []

for i in range(600):
    safe_action = genset_group_controller.generate_safe_action(action, reserve_available=True)
    genset_group.step(safe_action, reserve_available=True)
    obs_device = {'device_observations': genset_group.gather_observations()}
    genset_group_controller.update_controller_state(obs_device)
    obs_controller = genset_group_controller.gather_observations()

    genset_group_active_power.append(obs_controller['device_observations']['genset_group_active_power'])
    genset_1_power.append(obs_controller['device_observations']['gensets'][0]['device_observations']['active_power'])
    genset_1_avgpower.append(obs_controller['device_observations']['gensets'][0]['controller_state']['average_active_power'])
    genset_2_power.append(obs_controller['device_observations']['gensets'][1]['device_observations']['active_power'])
    genset_2_avgpower.append(obs_controller['device_observations']['gensets'][1]['controller_state']['average_active_power'])
    genset_3_power.append(obs_controller['device_observations']['gensets'][2]['device_observations']['active_power'])
    genset_3_avgpower.append(obs_controller['device_observations']['gensets'][2]['controller_state']['average_active_power'])

    assert obs_controller['device_observations']['gensets'][0]['device_observations']['active_power'] == obs_controller['device_observations']['gensets'][1]['device_observations']['active_power']         # Because they are all running and all have the same prime power, they always have the same active power even if one gets limited.
    assert obs_controller['device_observations']['gensets'][0]['device_observations']['active_power'] == obs_controller['device_observations']['gensets'][2]['device_observations']['active_power']         # Because they are all running and all have the same prime power, they always have the same active power even if one gets limited.

plt.plot(genset_1_power, label="Genset 1 active power")
plt.plot(genset_1_avgpower, label="Genset 1 average power", linestyle='dashed')
plt.plot(genset_2_power, label="Genset 2 active power")
plt.plot(genset_2_avgpower, label="Genset 2 average power", linestyle='dashed')
plt.plot(genset_3_power, label="Genset 3 active power")
plt.plot(genset_3_avgpower, label="Genset 3 average power", linestyle='dashed')
plt.xlabel("Time (minutes)")
plt.ylabel("Power (kW)")
plt.legend(["Genset 1 active power", "Genset 1 average power", "Genset 2 active power", "Genset 2 average power", "Genset 3 active power", "Genset 3 average power"])

plt.figure()
plt.plot(genset_group_active_power, label="Genset group active power")
plt.xlabel("Time (minutes)")
plt.ylabel("Group power (kW)")


assert obs_controller['device_observations']['gensets'][0]['device_observations']['active_power'] == 280        # Yet the genset cannot provide less than their minimum load of 120
assert obs_controller['device_observations']['gensets'][1]['device_observations']['active_power'] == 280
assert obs_controller['device_observations']['gensets'][2]['device_observations']['active_power'] == 280
assert obs_controller['device_observations']['genset_group_active_power'] == 280*3  

# Demand replay (true values)

In [ ]:
from demand import Demand
import datetime
import numpy as np
import pathlib


In [ ]:
date_time = datetime.datetime(2018, 6, 6, 6, 6, 6)


demand_params = {
    'const_params':{
        "mode": {'value': 'data', 'type': 'string'},
        "pred_time_step":  {'value': 1, 'type': 'integer'},
        "nb_pred_time_steps": {'value': 10, 'type': 'integer'},
        "normalisation_factor": {'value': 300, 'type': 'integer'},
        "forecast_model": {'value': 'ground_truth', 'type': 'string'},
    },
    'init_params': {
        "date_time": {'value': date_time, 'type': 'datetime'},
        "obs_type" : 'params'
    }
}

time_step = 1
demand_replayer = Demand(demand_params, time_step, real = True)

In [ ]:
date_time = datetime.datetime(2018, 6, 6, 6, 6, 6)
time_step = 1 # minute

# Get the first observation
demand_obs = demand_replayer.gather_observations()
date_time += datetime.timedelta(minutes = time_step)


# Spend 10 minutes
for i in range(10):
    old_demand_next = demand_obs['demand_next']
    old_demand_next_pred = demand_obs['demand_pred'][0]
    demand_replayer.step({'date_time': date_time})
    demand_obs = demand_replayer.gather_observations()
    date_time += datetime.timedelta(minutes = time_step)
    print(demand_obs)

    assert demand_obs['demand'] == old_demand_next # The demand should be the same as the previous demand next
    assert demand_obs['demand'] == old_demand_next_pred # The demand should be the same as the next step in the demand prediction



In [ ]:
old_demand_next_pred

In [ ]:
#Year end
date_time = datetime.datetime(2018, 12, 31, 23, 55, 0)

demand_params = {
    'const_params':{
        "mode": {'value': 'data', 'type': 'string'},
        "pred_time_step":  {'value': 1, 'type': 'integer'},
        "nb_pred_time_steps": {'value': 10, 'type': 'integer'},
        "normalisation_factor": {'value': 300, 'type': 'integer'},
        "forecast_model": {'value': 'ground_truth', 'type': 'string'},
    },
    'init_params': {
        "date_time": {'value': date_time, 'type': 'datetime'},
        "obs_type" : 'params'
    }
}

time_step = 1
demand_replayer = Demand(demand_params, time_step, real = True)



# Get the first observation
demand_obs= demand_replayer.gather_observations()
date_time += datetime.timedelta(minutes = time_step)


# Spend 10 minutes
for i in range(10):
    old_demand_next = demand_obs['demand_next']
    old_demand_next_pred = demand_obs['demand_pred'][0]

    demand_replayer.step({'date_time': date_time})
    demand_obs = demand_replayer.gather_observations()
    date_time += datetime.timedelta(minutes = time_step)
    print(demand_obs)

    assert demand_obs['demand'] == old_demand_next_pred # The demand should be the same as the next step in the demand prediction
    assert demand_obs['demand'] == old_demand_next # The demand should be the same as the previous demand next



Test the multiplication factor

In [ ]:
demand_replayer_1 = Demand(demand_params, time_step, real = True)

params_2 = demand_params.copy()
params_2['const_params']["normalisation_factor"]['value'] = 600

demand_replayer_2 = Demand(params_2, time_step, real = True)

In [ ]:
date_time = datetime.datetime(2018, 6, 6, 6, 6, 6)
time_step = 1 # minute

# Spend 10 minutes
for i in range(10):
    demand_replayer_1.step({'date_time': date_time})
    demand_obs_1 = demand_replayer_1.gather_observations()

    demand_replayer_2.step({'date_time': date_time})
    demand_obs_2 = demand_replayer_2.gather_observations()

    demand_ratio = demand_obs_2['demand']/demand_obs_1['demand']
    demand_pred_ratio = np.array(demand_obs_2['demand_pred'])/np.array(demand_obs_1['demand_pred'])

    assert demand_ratio == 2
    assert (demand_pred_ratio == np.ones(demand_params['const_params']["nb_pred_time_steps"]['value'])*2).all()

    date_time += datetime.timedelta(minutes = time_step)

print(demand_obs_1['demand_pred'])
print(demand_obs_2['demand_pred'])


# Demand replay (forecasts)

In [ ]:
from demand import Demand
import datetime
import numpy as np
import pathlib


In [ ]:


date_time = datetime.datetime(2018, 6, 6, 6, 6, 6)


demand_params = {
    'const_params':{
        "mode": {'value': 'data', 'type': 'string'},
        "pred_time_step":  {'value': 1, 'type': 'integer'},
        "nb_pred_time_steps": {'value': 10, 'type': 'integer'},
        "normalisation_factor": {'value': 300, 'type': 'integer'},
        "forecast_model": {'value': 'forecast', 'type': 'string'},
    },
    'init_params': {
        "date_time": {'value': date_time, 'type': 'datetime'},
        "obs_type" : 'params'
    }
}

time_step = 1

demand_replayer = Demand(demand_params, time_step, real = True)

In [ ]:
date_time = datetime.datetime(2018, 6, 6, 6, 6, 6)
time_step = 1 # minute

list_demands_from_66666 = [253.0768, 251.4286, 249.7804, 248.1322, 246.484, 247.9344, 249.3848, 250.8352, 252.2856, 253.736, 256.4614, 259.1868]
# Get the first observation
demand_replayer.step({'date_time': date_time})
demand_obs = demand_replayer.gather_observations()
assert demand_obs['demand'] == 253.0768 # The demand should be the same as the previous demand next
date_time += datetime.timedelta(minutes = time_step)


# Spend 10 minutes
for i in range(10):
    old_demand_next = demand_obs['demand_next']
    demand_replayer.step({'date_time': date_time})
    demand_obs = demand_replayer.gather_observations()
    date_time += datetime.timedelta(minutes = time_step)
    print(demand_obs)
    assert demand_obs['demand'] == old_demand_next # The demand should be the same as the previous demand next
    assert np.abs( demand_obs['demand'] - list_demands_from_66666[i+1]) < 10e-5


In [ ]:
demand_obs['demand']

In [ ]:
list_demands_from_66666[i+1]

In [ ]:
#Year end. In forecast mode, it stops at 11:50:00 instead of 23:59:00
date_time = datetime.datetime(2018, 12, 31, 11, 40, 0)

for i in range(20):
    demand_replayer.step({'date_time': date_time})
    demand_obs = demand_replayer.gather_observations()
    print(date_time)
    print(demand_obs)
    date_time += datetime.timedelta(minutes = time_step)

# Wind turbine

## Perlin wind turbine

In [ ]:
from windturbine import WindTurbine
from windturbine_controller import WindTurbineController
import datetime
import matplotlib.pyplot as plt
import numpy as np


wind_turbine_params = {
    'device': {
        'init_params': {
            'turbine_setpoint': {'value': 300, 'type': 'float'},
            'date_time': {'value': datetime.datetime(2018, 6, 6, 6, 6, 6), 'type': 'datetime'},
            'active': {'value': True, 'type': 'boolean'},
            'obs_type': 'params'
        },
        'const_params': {
            'pred_time_step': {'value': 1, 'type': 'integer'},
            'nb_pred_time_steps': {'value': 10, 'type': 'integer'},
            'mode': {'value': 'perlin', 'type': 'string'},
            'nominal_power': {'value': 500, 'type': 'float'},
            'perlin_params': {
                'average': {'value': 200, 'type': 'float'},
                'nb_octaves': {'value': 5, 'type': 'integer'},
                'octaves_step': {'value': 4, 'type': 'integer'},
                'period': {'value': 60*24, 'type': 'integer'},
                'persistence': {'value': 0.8, 'type': 'float'},
                'lacunarity': {'value': 2.5, 'type': 'float'},
                'repeat': {'value': 1024, 'type': 'integer'}
            }
        }
    },
    'controller': {
        'init_params': {
            'turbine_setpoint': {'value': 0, 'type': 'float'},
            'obs_type': 'params'
        },
        'const_params': {
            'power_max': {'value': 500, 'type': 'float'},
            'power_min': {'value': 0, 'type': 'float'},
        }
    }
}


time_step = 1



wind_turbine = WindTurbine(wind_turbine_params['device'], time_step=time_step, real=True)
wind_turbine_controller = WindTurbineController(wind_turbine_params, 1)

Test the demanded power control and the prediction

In [ ]:

date_time = datetime.datetime(2018, 6, 6, 6, 6, 0)
action = {
    'turbine_setpoint': 100,
    'date_time': date_time
}

safe_action = wind_turbine_controller.generate_safe_action(action)
assert safe_action['turbine_setpoint'] == action['turbine_setpoint']        # Action is in the safe range
wind_turbine.step(safe_action)
obs_1 = wind_turbine.gather_observations()
print(obs_1)

safe_action['date_time'] += datetime.timedelta(minutes = 1)

wind_turbine.step(safe_action)
obs_2 = wind_turbine.gather_observations()

print(obs_2)

avail_power_series = []
for i in range(wind_turbine_params['device']['const_params']['nb_pred_time_steps']['value']):
    safe_action['date_time'] += datetime.timedelta(minutes = 1)
    wind_turbine.step(safe_action)
    obs = wind_turbine.gather_observations()
    avail_power_series.append(obs['available_wind_power'])

assert obs_1['wind_power'] == action['turbine_setpoint']
assert obs_2['wind_power'] == action['turbine_setpoint']
assert obs_2['available_wind_power'] == obs_1['available_wind_power_pred'][0]
assert avail_power_series == obs_2['available_wind_power_pred']

Test the changes in the demanded control

In [ ]:
turbine_setpoint_series = []
safe_turbine_setpoint_series = []
avail_power_series = []
wind_power_series = []

date_time = datetime.datetime(2018, 6, 6, 6, 6, 0)

action = {
    'turbine_setpoint': 100,
    'date_time': date_time
}


for i in range(30):
    action['turbine_setpoint'] += 15
    action['date_time'] += datetime.timedelta(minutes = 1)
    safe_action = wind_turbine_controller.generate_safe_action(action)
    assert safe_action['turbine_setpoint'] == np.minimum(action['turbine_setpoint'], wind_turbine_params['controller']['const_params']['power_max']['value'])   
    wind_turbine.step(safe_action)
    obs = wind_turbine.gather_observations()
    avail_power_series.append(obs['available_wind_power'])
    turbine_setpoint_series.append(action['turbine_setpoint'])
    safe_turbine_setpoint_series.append(safe_action['turbine_setpoint'])
    wind_power_series.append(obs['wind_power'])
    assert obs['wind_power'] == np.minimum(action['turbine_setpoint'], obs['available_wind_power'])

plt.plot(turbine_setpoint_series, label="Turbine setpoint")
plt.plot(safe_turbine_setpoint_series, label="Safe turbine setpoint")
plt.plot(avail_power_series, label="Available power")
plt.plot(wind_power_series, label="Wind power")
plt.legend(["Turbine setpoint", "Safe turbine setpoint", "Available power", "Wind power"])

Check it cannot go below 0

In [ ]:
turbine_setpoint_series = []
safe_turbine_setpoint_series = []
avail_power_series = []
wind_power_series = []

date_time = datetime.datetime(2018, 6, 6, 6, 6, 0)

action = {
    'turbine_setpoint': 200,
    'date_time': date_time
}


for i in range(20):
    action['turbine_setpoint'] -= 15
    action['date_time'] += datetime.timedelta(minutes = 1)
    safe_action = wind_turbine_controller.generate_safe_action(action)
    assert safe_action['turbine_setpoint'] == np.clip(action['turbine_setpoint'], wind_turbine_params['controller']['const_params']['power_min']['value'], wind_turbine_params['controller']['const_params']['power_max']['value'])
    wind_turbine.step(safe_action)
    obs = wind_turbine.gather_observations()
    avail_power_series.append(obs['available_wind_power'])
    turbine_setpoint_series.append(action['turbine_setpoint'])
    safe_turbine_setpoint_series.append(safe_action['turbine_setpoint'])
    wind_power_series.append(obs['wind_power'])
    assert obs['wind_power'] == np.minimum(np.maximum(0, action['turbine_setpoint']), obs['available_wind_power'])
    

plt.plot(turbine_setpoint_series, label="Turbine setpoint")
plt.plot(safe_turbine_setpoint_series, label="Safe turbine setpoint")

plt.plot(avail_power_series, label="Available power")
plt.plot(wind_power_series, label="Wind power")
plt.legend(["Turbine setpoint", "Safe turbine setpoint", "Available power", "Wind power"])

Compute time for 30 days

In [ ]:
date_time = datetime.datetime(2018, 11, 6, 6, 6, 0)
import time


start_time = time.time()
for i in range(60*24*30):
    action['turbine_setpoint'] += 0
    action['date_time'] += datetime.timedelta(minutes = 1)
    safe_action = wind_turbine_controller.generate_safe_action(action)
    wind_turbine.step(safe_action)
    obs = wind_turbine.gather_observations()
run_time = time.time() - start_time

print("Run time for 30 days for Perlin wind turbine: {}".format(run_time))

Test average available wind power on a long term (one month) Note that it is expected for Perlin that the mean is a bit above the expected average because of the correction applied if the noise is lower than 0.

In [ ]:
date_time = datetime.datetime(2018, 11, 6, 6, 6, 0)

action = {
    'turbine_setpoint': 200,
    'date_time': date_time
}
for i in range(60*24*30):
    action['turbine_setpoint'] = 0
    action['date_time'] += datetime.timedelta(minutes = 1)
    safe_action = wind_turbine_controller.generate_safe_action(action)      # Does not matter, we just look for the available power
    wind_turbine.step(safe_action)
    obs = wind_turbine.gather_observations()
    avail_power_series.append(obs['available_wind_power'])

avail_power_array = np.array(avail_power_series)

avail_power_mean = np.mean(avail_power_array)
avail_power_std = np.std(avail_power_array)
avail_power_min = np.min(avail_power_array)
avail_power_max = np.max(avail_power_array)

print("Mean available power: {}; Standard deviation: {}".format(avail_power_mean, avail_power_std))
print("Minimum available power: {}; Maximum available power: {}".format(avail_power_min, avail_power_max))

plt.plot(avail_power_series, label="Available power")
plt.legend(["Available power"])


assert np.abs(avail_power_mean - wind_turbine_params['device']['const_params']['perlin_params']['average']['value']) < 10
assert avail_power_min >= 0 
assert avail_power_max <= wind_turbine_params['device']['const_params']['perlin_params']['average']['value'] + wind_turbine_params['device']['const_params']['perlin_params']['amplitude']['value']

In [ ]:
# Make histogram of available power with bins from 0 to 500 with step of 20
plt.hist(avail_power_array, bins=range(0, 500, 20));
plt.xlabel("Available power (kW)")
plt.ylabel("Count")

## Data

In [ ]:
from windturbine import WindTurbine
from windturbine_controller import WindTurbineController
import datetime
import matplotlib.pyplot as plt
import numpy as np


wind_turbine_params = {
    'device': {
        'init_params': {
            'turbine_setpoint': {'value': 300, 'type': 'float'},
            'date_time': {'value': datetime.datetime(2018, 6, 6, 6, 6, 6), 'type': 'datetime'},
            'active': {'value': True, 'type': 'boolean'},
            'obs_type': 'params'
        },
        'const_params': {
            'pred_time_step': {'value': 1, 'type': 'integer'},
            'nb_pred_time_steps': {'value': 30, 'type': 'integer'},
            'mode': {'value': 'data', 'type': 'string'},
            'nominal_power': {'value': 500, 'type': 'float'},
        }
    },
    'controller': {
        'init_params': {
            'turbine_setpoint': {'value': 0, 'type': 'float'},
            'obs_type': 'params'
        },
        'const_params': {
            'power_max': {'value': 500, 'type': 'float'},
            'power_min': {'value': 0, 'type': 'float'},
        }
    }
}


time_step = 1

wind_turbine = WindTurbine(wind_turbine_params['device'], time_step=time_step, real=True)
wind_turbine_controller = WindTurbineController(wind_turbine_params, 1)

In [ ]:
date_time = datetime.datetime(2018, 6, 6, 6, 6, 0)

action = {
    'turbine_setpoint': 100,
    'date_time': date_time
}

safe_action = wind_turbine_controller.generate_safe_action(action)      # Does not matter, we just look for the available power
wind_turbine.step(safe_action, date_time)
obs_1 = wind_turbine.gather_observations()
print(obs_1)

action['date_time'] += datetime.timedelta(minutes = 1)

safe_action = wind_turbine_controller.generate_safe_action(action)      # Does not matter, we just look for the available power
wind_turbine.step(safe_action, date_time)
obs_2 = wind_turbine.gather_observations()
print(obs_2)

# On the 6th of June at 6:06 AM, the power was higher than the turbine setpoint of 100kW --> the actual wind power is the setpoint.
assert obs_1['wind_power'] == action['turbine_setpoint']
assert obs_2['wind_power'] == action['turbine_setpoint']


assert np.abs(obs_1['available_wind_power'] - 250.72447217550298) < 10e-6
assert np.abs(obs_2['available_wind_power'] - 250.74924878438871) < 10e-6



In [ ]:
wind_turbine.get_available_wind_power(date_time)

In [ ]:
turbine_setpoint_series = []
avail_power_series = []
wind_power_series = []

date_time = datetime.datetime(2018, 6, 8, 5, 30, 0)
action = {
    'turbine_setpoint': 0,
    'date_time': date_time
}


for i in range(30):
    action['date_time'] += datetime.timedelta(minutes = 1)
    action['turbine_setpoint'] += 15
    safe_action = wind_turbine_controller.generate_safe_action(action)      # Does not matter, we just look for the available power
    wind_turbine.step(safe_action)
    obs = wind_turbine.gather_observations()
    avail_power_series.append(obs['available_wind_power'])
    turbine_setpoint_series.append(action['turbine_setpoint'])
    wind_power_series.append(obs['wind_power'])
    assert obs['wind_power'] == np.minimum(action['turbine_setpoint'], obs['available_wind_power'])

plt.plot(turbine_setpoint_series, label="Demanded power")
plt.plot(avail_power_series, label="Available power")
plt.plot(wind_power_series, label="Wind power")
plt.legend(["Demanded power", "Available power", "Wind power"])

In [ ]:
date_time = datetime.datetime(2018, 9, 6, 6, 6, 0)
import time

action['turbine_setpoint'] = 0
action['date_time'] = date_time


start_time = time.time()
for i in range(30*24*60):
    action['date_time'] += datetime.timedelta(minutes = 1)
    try:
        safe_action = wind_turbine_controller.generate_safe_action(action)      # Does not matter, we just look for the available power
        wind_turbine.step(safe_action)
        obs = wind_turbine.gather_observations()
    except:
        print("Error at time {}".format(date_time))
run_time = time.time() - start_time

print("Run time for 30 days: {}".format(run_time))

In [ ]:
# Test the predictions
import os

for month in range(1, 13):
    action['date_time'] = datetime.datetime(2018, month, 6, 2, 46, 3)
    safe_action = wind_turbine_controller.generate_safe_action(action)      # Does not matter, we just look for the available power
    wind_turbine.step(safe_action, date_time)
    obs = wind_turbine.gather_observations()
    prediction = obs['available_wind_power_pred']
    available_power_list = []
    for i in range(0, (wind_turbine.nb_pred_time_steps+1) * wind_turbine.pred_time_step, wind_turbine.pred_time_step):
        action['date_time'] = date_time + datetime.timedelta(minutes=i*wind_turbine.pred_time_step)
        safe_action = wind_turbine_controller.generate_safe_action(action)      # Does not matter, we just look for the available power
        wind_turbine.step(safe_action)
        obs = wind_turbine.gather_observations()
        available_power_list.append(obs['available_wind_power'])
    plt.figure()
    plt.plot(range(0, (wind_turbine.nb_pred_time_steps+1)*wind_turbine.pred_time_step, wind_turbine.pred_time_step), available_power_list, label='Available power')
    plt.plot(range(wind_turbine.pred_time_step, (wind_turbine.nb_pred_time_steps+1)*wind_turbine.pred_time_step, wind_turbine.pred_time_step), prediction, label='Prediction')
    plt.legend()
    plt.ylim(-1, 501)
    plt.xlabel("Time (minutes)")
    plt.ylabel("Power (kW)")
    plt.title("Prediction date: {}, time step: {} minutes.".format(date_time, wind_turbine.pred_time_step))



# Microgrid without turbine

## Test general functions with battery shield only

In [ ]:
from microgrid import MicroGrid
from microgrid_controller import MicroGridController
import datetime
import numpy as np
import pathlib
import pprint

In [ ]:
external_world_params = {
    "init_date_time": {"value": datetime.datetime(2018, 6, 6, 6, 6, 0), "type": "datetime"},
    "time_step": {"value": 1, "type": "integer"}
}

genset_params = {
    'device': {
        'const_params': {
            'alpha_g': {'value': 0.25, 'type': 'float'}, 'beta_g': {'value': 10, 'type': 'float'},
            'prime_power_rating': {'value': 400, 'type': 'float'}, 'temp_overload_factor': {'value': 1.1, 'type': 'float'},
            'time_step': {'value': 1, 'type': 'int'}, 'noise_level': {'value': 'none', 'type': 'string'},
            'noise_range_percentage': {'value': 0.2, 'type': 'float'}
        },
        'init_params': {
            'running': {'value': False, 'type': 'bool'}, 'init_power_setpoint': {'value': 300, 'type': 'float'}
        }
    },
    'controller': {
        'const_params': {
            'minimum_run_time': {'value': 30, 'type': 'integer'}, 'minimum_load': {'value': 120, 'type': 'float'},
            'warmup_power': {'value': 100, 'type': 'float'}, 'cooldown_power': {'value': 0, 'type': 'float'},
            'warmup_time': {'value': 3, 'type': 'integer'}, 'cooldown_time': {'value': 6, 'type': 'integer'},
            'average_active_power_constraint': {'value': True, 'type': 'boolean'},
            'average_active_power_window': {'value': 2880, 'type': 'integer'},
            'average_active_power_limit': {'value': 0.7, 'type': 'float'}
        },
        'init_params': {
            'status': {'value': 'off', 'type': 'string', 'elements': ['off', 'running', 'warmup', 'cooldown']},
            'time_since_warmup': {'value': 0, 'type': 'integer'}, 'time_since_cooldown': {'value': 0, 'type': 'integer'},
            'time_since_start': {'value': 0, 'type': 'integer'}, 'average_active_power': {'value': 100, 'type': 'float'}
        }
    }
}

genset_group_params = {
    'device': {
        'const_params': {'n_gensets': {'value': 3, 'type': 'integer'}},
        'init_params': {'gensets': [genset_params] * 3, 'obs_type': 'params'}
    },
    'controller': {
        'const_params': {'priority_order': {'value': True, 'type': 'boolean'}},
        'init_params': {'obs_type': 'params'}
    }
}

battery_params = {
    "device": {
        "const_params": {
            "P_nom": {"value": 600, "type": "float"},
            "soc_min": {"value": 0, "type": "float"},
            "soc_max": {"value": 1, "type": "float"},
            "I_nom": {"value": 672, "type": "float"},
            "Q_max": {"value": 672, "type": "float"},
            "eta_charge": {"value": 0.95, "type": "float"},
            "R": {"value": 0.039375, "type": "float"},
            "A": {"value": 39.6, "type": "float"},
            "B": {"value": 929.16, "type": "float"},
            "alpha_d": {"value": 5, "type": "float"},
            "beta": {"value": 1, "type": "float"},
            "degradation_cost_type": {"value": "cycle_based", "type": "string"},
            "buffer_size": {"value": 30, "type": "integer"},
            "noise_level": {"value": "none", "type": "string"},
            "noise_range_percentage": {"value": 0.2, "type": "float"},
        },
        "init_params": {
            "soc": {"value": 0.5, "type": "float", "min": 0.1, "max": 0.9},
            "p_grid": {"value": 0, "type": "float"},
            "obs_type": 'params'
        },
    },
    "controller": {
        "const_params": {
            "soc_max_norm": {"value": 0.9, "type": "float"},
            "soc_min_norm": {"value": 0.1, "type": "float"},
            "soc_max_res": {"value": 0.95, "type": "float"},
            "soc_min_res": {"value": 0.05, "type": "float"},
        },
        "init_params": {
            "obs_type": 'params'
        },
    },
}


demand_params = {
    'controller': {'const_params':{}, 'init_params':{'obs_type': 'params'}},
    'device': {
        'const_params': {
            "mode": {'value': 'data', 'type': 'string'},
            "pred_time_step":  {'value': 1, 'type': 'integer'},
            "nb_pred_time_steps": {'value': 10, 'type': 'integer'},
            "normalisation_factor": {'value': 300, 'type': 'integer'},
            "forecast_model": {'value': 'forecast', 'type': 'string'},
        },
        'init_params': {
            "obs_type": 'params',
        }
    }
}


microgrid_params = {
    "device": {
        "const_params": {
        },
        "init_params": {
            "external_world": external_world_params,
            "genset_group": genset_group_params,
            "battery": battery_params,
            "demand": demand_params,
            "obs_type": 'params'
        },
    },
    "controller": {
        "const_params": {
            "check_initialization": {"value": True, "type": "boolean"},
            "init_check_steps": {"value": 8, "type": "integer"},
            "init_check_loops": {"value": 3, "type": "integer"},
            "wind_turbine_priority": {"value": True, "type": "boolean"},
            "check_balance": {"value": False, "type": "boolean"},                       # No check balance --> No predictive shield on generator status if steady demand in next 10 time step is irrecoverable without reserve
            "check_reserve": {"value": False, "type": "boolean"},                       # No check reserve --> No predictive shield on generator status if worst case demand in next 10 time step is irrecoverable with reserve
            "adjust_battery": {"value": True, "type": "boolean"},
            "conservativeness_coeff": {"value": 1, "type": "float"},
            "min_available_wind_power": {"value": 0, "type": "float"},
            "max_available_wind_power_drop_step": {"value": 0.18, "type": "float"},
            "max_demand": {"value": 570, "type": "float"},
            "max_demand_increase_step": {"value": 80, "type": "float"},
        },
        "init_params": {
            "obs_type": 'params',
        },
    }
}

time_step = 1


In [ ]:
microgrid = MicroGrid(microgrid_params['device'], time_step, real=True)
microgrid_controller = MicroGridController(microgrid_params, time_step)
microgrid_controller.update_controller_state({'device_observations': microgrid.gather_observations(), 'controller_state': {}})  # Initialize the controller with the current state of the microgrid  

pprint.pprint(microgrid.gather_observations())

In [ ]:
microgrid_controller.update_controller_state({'device_observations': microgrid.gather_observations(), 'controller_state': {}})  # Initialize the controller with the current state of the microgrid  
microgrid_controller.gather_observations()

At initialization, there is no genset running and the battery charge is at 50%.
At the first step, even if we send a command of 0 to the battery, the battery will be asked to provide to cover the demand.

In [ ]:
action = {
    "genset_group": {"status_change": "none"},
    "battery": {"p_grid": 0}
}

safe_action = microgrid_controller.generate_safe_action(action)
microgrid.step(safe_action, verbose = True)
microgrid_controller.update_controller_state({'device_observations': microgrid.gather_observations(), 'controller_state': {}})  # Update the controller with the current state of the microgrid
observations = microgrid_controller.gather_observations()

assert observations['device_observations']['genset_group']['device_observations']['genset_group_active_power'] == 0
assert observations['device_observations']['demand']['device_observations']['demand'] == 251.4286
assert observations['device_observations']['battery']['device_observations']['p_grid'] == observations['device_observations']['demand']['device_observations']['demand']
assert observations['device_observations']['balance'] == 0

Turn on a genset: the first genset will get in warmup, the battery will cover what is missing

In [ ]:
action = {
    "genset_group": {"status_change": "start_next"},
    "battery": {"p_grid": 0}
}

safe_action = microgrid_controller.generate_safe_action(action)
microgrid.step(safe_action, verbose = True)
microgrid_controller.update_controller_state({'device_observations': microgrid.gather_observations(), 'controller_state': {}})  # Update the controller with the current state of the microgrid
observations = microgrid_controller.gather_observations()

assert observations['device_observations']['genset_group']['device_observations']['genset_group_active_power'] == genset_params['controller']['const_params']['warmup_power']['value']
assert observations['device_observations']['battery']['device_observations']['p_grid'] == observations['device_observations']['demand']['device_observations']['demand'] - genset_params['controller']['const_params']['warmup_power']['value']
assert observations['device_observations']['balance'] == 0

Do 3 steps: the genset is now controllable, the battery should stop providing power.

In [ ]:
action = {
    "genset_group": {"status_change": "none"},
    "battery": {"p_grid": 0}
}

safe_action = microgrid_controller.generate_safe_action(action)
microgrid.step(safe_action, verbose = True)
microgrid_controller.update_controller_state({'device_observations': microgrid.gather_observations(), 'controller_state': {}})  # Update the controller with the current state of the microgrid
observations = microgrid_controller.gather_observations()
assert observations['device_observations']['genset_group']['device_observations']['genset_group_active_power'] == genset_params['controller']['const_params']['warmup_power']['value']
assert np.abs(observations['device_observations']['balance']) < 10e-6

safe_action = microgrid_controller.generate_safe_action(action)
microgrid.step(safe_action, verbose = True)
microgrid_controller.update_controller_state({'device_observations': microgrid.gather_observations(), 'controller_state': {}})  # Update the controller with the current state of the microgrid
observations = microgrid_controller.gather_observations()
assert observations['device_observations']['genset_group']['device_observations']['genset_group_active_power'] == genset_params['controller']['const_params']['warmup_power']['value']
assert np.abs(observations['device_observations']['balance']) < 10e-6


## Should be done being in warmup
safe_action = microgrid_controller.generate_safe_action(action)
microgrid.step(safe_action, verbose = True)
microgrid_controller.update_controller_state({'device_observations': microgrid.gather_observations(), 'controller_state': {}})  # Update the controller with the current state of the microgrid
observations = microgrid_controller.gather_observations()

assert np.abs(observations['device_observations']['battery']['device_observations']['p_grid']) < 10e-2 # Battery is off
assert np.abs(observations['device_observations']['balance']) < 10e-2
assert np.abs(observations['device_observations']['genset_group']['device_observations']['genset_group_active_power'] - observations['device_observations']['demand']['device_observations']['demand']) < 10e-2   


Command to use some battery power. The genset produces less.

In [ ]:
action = {
    "genset_group": {"status_change": "none"},
    "battery": {"p_grid": 100}
}

safe_action = microgrid_controller.generate_safe_action(action)
microgrid.step(safe_action, verbose = True)
microgrid_controller.update_controller_state({'device_observations': microgrid.gather_observations(), 'controller_state': {}})  # Update the controller with the current state of the microgrid
observations = microgrid_controller.gather_observations()
assert np.abs(observations['device_observations']['genset_group']['device_observations']['genset_group_active_power'] - (observations['device_observations']['demand']['device_observations']['demand'] - 100) ) < 10e-2
assert np.abs(observations['device_observations']['balance']) < 10e-6
assert np.abs(observations['device_observations']['battery']['device_observations']['p_grid'] - 100) < 10e-2


Command to use more battery power. The generator reaches its minimum load, therefore the battery adapts.

In [ ]:
action = {
    "genset_group": {"status_change": "none"},
    "battery": {"p_grid": 200}
}
safe_action = microgrid_controller.generate_safe_action(action)
microgrid.step(safe_action, verbose = True)
microgrid_controller.update_controller_state({'device_observations': microgrid.gather_observations(), 'controller_state': {}})  # Update the controller with the current state of the microgrid
observations = microgrid_controller.gather_observations()

assert np.abs(observations['device_observations']['genset_group']['device_observations']['genset_group_active_power'] - genset_params['controller']['const_params']['minimum_load']['value'] ) < 10e-2

assert np.abs(observations['device_observations']['battery']['device_observations']['p_grid'] - (observations['device_observations']['demand']['device_observations']['demand'] - genset_params['controller']['const_params']['minimum_load']['value'])) < 10e-2
assert np.abs(observations['device_observations']['balance']) < 10e-6


Try to turn off the genset, will be refused because it has not been running for long enough. As battery power is 0, the genset will take all the load.

In [ ]:
action = {
    "genset_group": {"status_change": "stop_last"},
    "battery": {"p_grid": 0}
}

safe_action = microgrid_controller.generate_safe_action(action)
microgrid.step(safe_action, verbose = True)
microgrid_controller.update_controller_state({'device_observations': microgrid.gather_observations(), 'controller_state': {}})  # Update the controller with the current state of the microgrid
observations = microgrid_controller.gather_observations()

assert observations['device_observations']['genset_group']['device_observations']['gensets'][0]['device_observations']['running']
assert observations['device_observations']['genset_group']['device_observations']['gensets'][0]['controller_state']['status'] == 'running'
assert np.abs(observations['device_observations']['battery']['device_observations']['p_grid']) < 10e-2 # Battery is off
assert np.abs(observations['device_observations']['balance']) < 10e-2
assert np.abs(observations['device_observations']['genset_group']['device_observations']['genset_group_active_power'] - observations['device_observations']['demand']['device_observations']['demand']) < 10e-2   



Run the gensets for 30 minutes, then turn it off. The battery should take the whole load.

In [ ]:
action = {
    "genset_group": {"status_change": "none"},
    "battery": {"p_grid": 0}
}

for i in range(30):
    safe_action = microgrid_controller.generate_safe_action(action)
    microgrid.step(safe_action, verbose = True)
    microgrid_controller.update_controller_state({'device_observations': microgrid.gather_observations(), 'controller_state': {}})  # Update the controller with the current state of the microgrid
    observations = microgrid_controller.gather_observations()
    assert observations['device_observations']['genset_group']['device_observations']['gensets'][0]['device_observations']['running']
    assert observations['device_observations']['genset_group']['device_observations']['gensets'][0]['controller_state']['status'] == 'running'
    assert np.abs(observations['device_observations']['battery']['device_observations']['p_grid']) < 10e-2 # Battery is off
    assert np.abs(observations['device_observations']['balance']) < 10e-2
    assert np.abs(observations['device_observations']['genset_group']['device_observations']['genset_group_active_power'] - observations['device_observations']['demand']['device_observations']['demand']) < 10e-2   


action = {
    "genset_group": {"status_change": "stop_last"},
    "battery": {"p_grid": 0}
}

safe_action = microgrid_controller.generate_safe_action(action)
microgrid.step(safe_action, verbose = True)
microgrid_controller.update_controller_state({'device_observations': microgrid.gather_observations(), 'controller_state': {}})  # Update the controller with the current state of the microgrid
observations = microgrid_controller.gather_observations()

assert observations['device_observations']['genset_group']['device_observations']['gensets'][0]['controller_state']['status'] == 'cooldown' # The genset is in cooldown
assert observations['device_observations']['genset_group']['device_observations']['gensets'][0]['device_observations']['running']       # Cooldown is still running
assert np.abs(observations['device_observations']['battery']['device_observations']['p_grid'] -  observations['device_observations']['demand']['device_observations']['demand']) < 10e-2 # Battery takes all the load

assert np.abs(observations['device_observations']['balance']) < 10e-2



Run on the battery until after it cannot provide anymore. Plotting the battery SOC and the demand.

In [ ]:
import matplotlib.pyplot as plt

action = {
    "genset_group": {"status_change": "none"},
    "battery": {"p_grid": 0}
}

battery_soc = []
battery_power = []
balance = []

for i in range(150):
    safe_action = microgrid_controller.generate_safe_action(action)
    microgrid.step(safe_action, verbose = True)
    microgrid_controller.update_controller_state({'device_observations': microgrid.gather_observations(), 'controller_state': {}})  # Update the controller with the current state of the microgrid
    observations = microgrid_controller.gather_observations()
    battery_soc.append(observations['device_observations']['battery']['device_observations']['soc'])
    battery_power.append(observations['device_observations']['battery']['device_observations']['p_grid'])
    balance.append(observations['device_observations']['balance'])

plt.plot(battery_soc)
plt.xlabel("Time (min)")
plt.ylabel("Battery SOC")

plt.figure()
plt.plot(battery_power)
plt.xlabel("Time (min)")
plt.ylabel("Battery power")

plt.figure()
plt.plot(balance)
plt.xlabel("Time (min)")
plt.ylabel("Balance")

assert observations['device_observations']['genset_group']['device_observations']['genset_group_active_power'] == 0
assert observations['device_observations']['battery']['device_observations']['p_grid'] <= 0.00001
assert observations['device_observations']['balance'] == - observations['device_observations']['demand']['device_observations']['demand']  
assert observations['device_observations']['battery']['device_observations']['soc'] == battery_params['controller']['const_params']['soc_min_res']['value'] # The SOC is at the minimum - it has used the reserve



Start a generator and run it until it passes the warmup time. The generator will take care of the demand, and will also charge the battery because it is under the reserve limit.

In [ ]:
action = {
    "genset_group": {"status_change": "start_next"},
    "battery": {"p_grid": 0}
}

safe_action = microgrid_controller.generate_safe_action(action)
microgrid.step(safe_action, verbose = False)
microgrid_controller.update_controller_state({'device_observations': microgrid.gather_observations(), 'controller_state': {}})  # Update the controller with the current state of the microgrid
observations = microgrid_controller.gather_observations()


action = {
    "genset_group": {"status_change": "none"},
    "battery": {"p_grid": 0}
}


for i in range(4):          # Pass warmup
    safe_action = microgrid_controller.generate_safe_action(action)
    microgrid.step(safe_action, verbose = False)
    microgrid_controller.update_controller_state({'device_observations': microgrid.gather_observations(), 'controller_state': {}})  # Update the controller with the current state of the microgrid
    observations = microgrid_controller.gather_observations()


assert np.abs(observations['device_observations']['genset_group']['device_observations']['genset_group_active_power'] - genset_params['device']['const_params']['prime_power_rating']['value']) < 10e-2        # Genset is working at maximum
assert np.abs(observations['device_observations']['battery']['device_observations']['p_grid'] + genset_params['device']['const_params']['prime_power_rating']['value'] - observations['device_observations']['demand']['device_observations']['demand']) < 10e-2 # Battery is charging with what is left from the genset after removing demand
assert np.abs(observations['device_observations']['balance']) < 10e-2

print("Battery SOC: {}".format(observations['device_observations']['battery']['device_observations']['soc']))



Keep on going until battery is charged at the reserve limit. The battery stops charging and the genset provides for the demand.

In [ ]:
action = {
    "genset_group": {"status_change": "none"},
    "battery": {"p_grid": 0}
}


for i in range(35):     # Enough to charge the battery
    safe_action = microgrid_controller.generate_safe_action(action)
    microgrid.step(safe_action, verbose = False)
    microgrid_controller.update_controller_state({'device_observations': microgrid.gather_observations(), 'controller_state': {}})  # Update the controller with the current state of the microgrid
    observations = microgrid_controller.gather_observations()
    
    print("SOC: ", observations['device_observations']['battery']['device_observations']['soc'])


assert np.abs(observations['device_observations']['genset_group']['device_observations']['genset_group_active_power'] - observations['device_observations']['demand']['device_observations']['demand']) < 10e-2        # Genset is providing for the demand
assert np.abs(observations['device_observations']['battery']['device_observations']['p_grid'] ) < 10e-2 # Battery is idle
assert np.abs(observations['device_observations']['balance']) < 10e-2




Start a new genset to have more power. The genset gets in warmup, the other genset produces less power.

In [ ]:
action = {
    "genset_group": {"status_change": "start_next"},
    "battery": {"p_grid": 0}
}

safe_action = microgrid_controller.generate_safe_action(action)
microgrid.step(safe_action, verbose = False)
microgrid_controller.update_controller_state({'device_observations': microgrid.gather_observations(), 'controller_state': {}})  # Update the controller with the current state of the microgrid
observations = microgrid_controller.gather_observations()

assert observations['device_observations']['genset_group']['device_observations']['gensets'][1]['device_observations']['active_power'] == genset_params['controller']['const_params']['warmup_power']['value']       # Genset is working at maximum
assert np.abs(observations['device_observations']['genset_group']['device_observations']['genset_group_active_power'] - observations['device_observations']['demand']['device_observations']['demand']) < 10e-2        # Genset is providing for the demand
assert np.abs(observations['device_observations']['battery']['device_observations']['p_grid'] ) < 10e-2 # Battery is idle
assert np.abs(observations['device_observations']['balance']) < 10e-2

Charge the battery with asking for the maximum. It should be reduced to get only what both gensets can produce at max power (without overload). The battery will charge until SOC max, the balance will stay at 0.

In [ ]:
action = {
    "genset_group": {"status_change": "none"},
    "battery": {"p_grid": -600}
}

genset_power = []
battery_soc = []
battery_power = []
balance = []


for i in range(20):
    safe_action = microgrid_controller.generate_safe_action(action)
    microgrid.step(safe_action, verbose = False)
    microgrid_controller.update_controller_state({'device_observations': microgrid.gather_observations(), 'controller_state': {}})  
    observations = microgrid_controller.gather_observations()    
    battery_soc.append(observations['device_observations']['battery']['device_observations']['soc'])
    battery_power.append(observations['device_observations']['battery']['device_observations']['p_grid'])
    balance.append(observations['device_observations']['balance'])
    genset_power.append(observations['device_observations']['genset_group']['device_observations']['genset_group_active_power'])

# After 20 steps, battery not completely charged, so gensets are working at maximum
assert np.abs(observations['device_observations']['genset_group']['device_observations']['genset_group_active_power'] - 2 * genset_params['device']['const_params']['prime_power_rating']['value']) < 10e-2        
assert np.abs(observations['device_observations']['battery']['device_observations']['p_grid'] + observations['device_observations']['genset_group']['device_observations']['genset_group_active_power'] - observations['device_observations']['demand']['device_observations']['demand']) < 10e-2
assert np.abs(observations['device_observations']['balance']) < 10e-2
assert observations['device_observations']['battery']['device_observations']['soc'] < battery_params['controller']['const_params']['soc_max_norm']['value'] 

for i in range(100):
    safe_action = microgrid_controller.generate_safe_action(action)
    microgrid.step(safe_action, verbose = False)
    microgrid_controller.update_controller_state({'device_observations': microgrid.gather_observations(), 'controller_state': {}}) 
    observations = microgrid_controller.gather_observations()    
    battery_soc.append(observations['device_observations']['battery']['device_observations']['soc'])
    battery_power.append(observations['device_observations']['battery']['device_observations']['p_grid'])
    balance.append(observations['device_observations']['balance'])
    genset_power.append(observations['device_observations']['genset_group']['device_observations']['genset_group_active_power'])

# After 100 steps, battery is charged at value without reserve. Gensets fill the demand. The battery is idle.

assert np.abs(observations['device_observations']['genset_group']['device_observations']['genset_group_active_power'] - observations['device_observations']['demand']['device_observations']['demand']) < 10e-2   
assert np.abs(observations['device_observations']['battery']['device_observations']['p_grid']) < 10e-2   
assert np.abs(observations['device_observations']['balance']) < 10e-2
assert np.abs(observations['device_observations']['battery']['device_observations']['soc'] - battery_params['controller']['const_params']['soc_max_norm']['value']) < 10e-2
assert np.all(np.abs(np.array(balance)) <= 1e-6)

plt.plot(battery_soc)
plt.xlabel("Time (min)")
plt.ylabel("Battery SOC")

plt.figure()
plt.plot(battery_power)
plt.xlabel("Time (min)")
plt.ylabel("Battery power")

plt.figure()
plt.plot(balance)
plt.xlabel("Time (min)")
plt.ylabel("Balance")

plt.figure()
plt.plot(genset_power)
plt.xlabel("Time (min)")
plt.ylabel("Genset power")

## Test complete shield - checking recoverability

In [ ]:
from microgrid import MicroGrid
from microgrid_controller import MicroGridController
import datetime
import pathlib
import numpy as np

We turn on the balance check test. We start with all gensets OFF. We double the demand, so the battery will discharge fast.
Note that the _reserve_ check test is off. The gensets initial average active power is set to 278, close to the 280 limit.

In [ ]:
external_world_params = {
    "init_date_time": {"value": datetime.datetime(2018, 6, 6, 5, 35, 0), "type": "datetime"},
    "time_step": {"value": 1, "type": "integer"}
}

genset_params = {
    'device': {
        'const_params': {
            'alpha_g': {'value': 0.25, 'type': 'float'}, 'beta_g': {'value': 10, 'type': 'float'},
            'prime_power_rating': {'value': 400, 'type': 'float'}, 'temp_overload_factor': {'value': 1.1, 'type': 'float'},
            'time_step': {'value': 1, 'type': 'int'}, 'noise_level': {'value': 'none', 'type': 'string'},
            'noise_range_percentage': {'value': 0.2, 'type': 'float'}
        },
        'init_params': {
            'running': {'value': False, 'type': 'bool'}, 'init_power_setpoint': {'value': 300, 'type': 'float'}
        }
    },
    'controller': {
        'const_params': {
            'minimum_run_time': {'value': 30, 'type': 'integer'}, 'minimum_load': {'value': 120, 'type': 'float'},
            'warmup_power': {'value': 100, 'type': 'float'}, 'cooldown_power': {'value': 0, 'type': 'float'},
            'warmup_time': {'value': 3, 'type': 'integer'}, 'cooldown_time': {'value': 6, 'type': 'integer'},
            'average_active_power_constraint': {'value': True, 'type': 'boolean'},
            'average_active_power_window': {'value': 2880, 'type': 'integer'},
            'average_active_power_limit': {'value': 0.7, 'type': 'float'}
        },
        'init_params': {
            'status': {'value': 'off', 'type': 'string', 'elements': ['off', 'running', 'warmup', 'cooldown']},
            'time_since_warmup': {'value': 0, 'type': 'integer'}, 'time_since_cooldown': {'value': 0, 'type': 'integer'},
            'time_since_start': {'value': 0, 'type': 'integer'}, 'average_active_power': {'value': 278, 'type': 'float'}
        }
    }
}

genset_group_params = {
    'device': {
        'const_params': {'n_gensets': {'value': 3, 'type': 'integer'}},
        'init_params': {'gensets': [genset_params] * 3, 'obs_type': 'params'}
    },
    'controller': {
        'const_params': {'priority_order': {'value': True, 'type': 'boolean'}},
        'init_params': {'obs_type': 'params'}
    }
}

battery_params = {
    "device": {
        "const_params": {
            "P_nom": {"value": 600, "type": "float"},
            "soc_min": {"value": 0, "type": "float"},
            "soc_max": {"value": 1, "type": "float"},
            "I_nom": {"value": 672, "type": "float"},
            "Q_max": {"value": 672, "type": "float"},
            "eta_charge": {"value": 0.95, "type": "float"},
            "R": {"value": 0.039375, "type": "float"},
            "A": {"value": 39.6, "type": "float"},
            "B": {"value": 929.16, "type": "float"},
            "alpha_d": {"value": 5, "type": "float"},
            "beta": {"value": 1, "type": "float"},
            "degradation_cost_type": {"value": "cycle_based", "type": "string"},
            "buffer_size": {"value": 30, "type": "integer"},
            "noise_level": {"value": "none", "type": "string"},
            "noise_range_percentage": {"value": 0.2, "type": "float"},
        },
        "init_params": {
            "soc": {"value": 0.5, "type": "float", "min": 0.1, "max": 0.9},
            "p_grid": {"value": 0, "type": "float"},
            "obs_type": 'params'
        },
    },
    "controller": {
        "const_params": {
            "soc_max_norm": {"value": 0.9, "type": "float"},
            "soc_min_norm": {"value": 0.1, "type": "float"},
            "soc_max_res": {"value": 0.95, "type": "float"},
            "soc_min_res": {"value": 0.05, "type": "float"},
        },
        "init_params": {
            "obs_type": 'params'
        },
    },
}


demand_params = {
    'controller': {'const_params':{}, 'init_params':{'obs_type': 'params'}},
    'device': {
        'const_params': {
            "mode": {'value': 'data', 'type': 'string'},
            "pred_time_step": {'value': 1, 'type': 'integer'},
            "nb_pred_time_steps": {'value': 10, 'type': 'integer'},
            "normalisation_factor": {'value': 600, 'type': 'integer'},               # Multiply demand by 2
            "forecast_model": {'value': 'forecast', 'type': 'string'},
        },
        'init_params': {
            "obs_type": 'params',
        }
    }
}


microgrid_params = {
    "device": {
        "const_params": {
        },
        "init_params": {
            "external_world": external_world_params,
            "genset_group": genset_group_params,
            "battery": battery_params,
            "demand": demand_params,
            "obs_type": 'params'
        },
    },
    "controller": {
        "const_params": {
            "check_initialization": {"value": True, "type": "boolean"},
            "init_check_steps": {"value": 8, "type": "integer"},
            "init_check_loops": {"value": 3, "type": "integer"},
            "wind_turbine_priority": {"value": True, "type": "boolean"},
            "check_balance": {"value": True, "type": "boolean"},                       # !!! Check balance is ON --> Predictive shield on generator status if steady demand in next 10 time step is irrecoverable without reserve
            "check_reserve": {"value": False, "type": "boolean"},                       # Still no check reserve --> No predictive shield on generator status if worst case demand in next 10 time step is irrecoverable with reserve
            "adjust_battery": {"value": True, "type": "boolean"},
            "conservativeness_coeff": {"value": 1, "type": "float"},
            "min_available_wind_power": {"value": 0, "type": "float"},
            "max_available_wind_power_drop_step": {"value": 0.18, "type": "float"},
            "max_demand": {"value": 570, "type": "float"},
            "max_demand_increase_step": {"value": 80, "type": "float"},
        },
        "init_params": {
            "obs_type": 'params',
        },
    }
}

time_step = 1


In [ ]:
microgrid = MicroGrid(microgrid_params['device'], time_step, real=True)
microgrid_controller = MicroGridController(microgrid_params, time_step)
microgrid_controller.update_controller_state({'device_observations': microgrid.gather_observations(), 'controller_state': {}})  # Initialize the controller with the current state of the microgrid  

Run without sending a "start_next" action. The battery should cover while it can but when its SOC is too low, a genset should be turned on automatically by the shield, and after warmup, it should help. Then, once the battery is empty, a second genset should turn on. Never during the whole process should the balance be negative nor the generators go overload.

In [ ]:
import matplotlib.pyplot as plt

action = {
    "genset_group": {"status_change": "none"},
    "battery": {"p_grid": 0}
}

genset_power = []
battery_soc = []
battery_power = []
balance = []
genset_0_power = []
genset_1_power = []
genset_2_power = []
genset_0_overload = []
genset_1_overload = []
genset_2_overload = []
genset_0_avgpower = []
genset_1_avgpower = []
genset_2_avgpower = []
demand = []
number_genset_running = []
number_genset_warmup = []



for i in range(60):
    safe_action = microgrid_controller.generate_safe_action(action)
    microgrid.step(safe_action)
    microgrid_controller.update_controller_state({'device_observations': microgrid.gather_observations(), 'controller_state': {}}) 
    observations = microgrid_controller.gather_observations()  
    battery_soc.append(observations['device_observations']['battery']['device_observations']['soc'])
    battery_power.append(observations['device_observations']['battery']['device_observations']['p_grid'])
    balance.append(observations['device_observations']['balance'])
    genset_power.append(observations['device_observations']['genset_group']['device_observations']['genset_group_active_power'])
    genset_0_power.append(observations['device_observations']['genset_group']['device_observations']['gensets'][0]['device_observations']['active_power'])
    genset_1_power.append(observations['device_observations']['genset_group']['device_observations']['gensets'][1]['device_observations']['active_power'])
    genset_2_power.append(observations['device_observations']['genset_group']['device_observations']['gensets'][2]['device_observations']['active_power'])
    genset_0_overload.append(1 if observations['device_observations']['genset_group']['device_observations']['gensets'][0]['device_observations']['overload'] else 0)
    genset_1_overload.append(1 if observations['device_observations']['genset_group']['device_observations']['gensets'][1]['device_observations']['overload'] else 0)
    genset_2_overload.append(1 if observations['device_observations']['genset_group']['device_observations']['gensets'][2]['device_observations']['overload'] else 0)
    genset_0_avgpower.append(observations['device_observations']['genset_group']['device_observations']['gensets'][0]['controller_state']['average_active_power'])
    genset_1_avgpower.append(observations['device_observations']['genset_group']['device_observations']['gensets'][1]['controller_state']['average_active_power'])
    genset_2_avgpower.append(observations['device_observations']['genset_group']['device_observations']['gensets'][2]['controller_state']['average_active_power'])
    demand.append(observations['device_observations']['demand']['device_observations']['demand'])
    number_genset_running.append(len(observations['device_observations']['genset_group']['device_observations']['config_lists']['running_gensets_ids']))
    number_genset_warmup.append(len(observations['device_observations']['genset_group']['device_observations']['config_lists']['warmup_gensets_ids']))



plt.plot(battery_soc)
plt.xlabel("Time (min)")
plt.ylabel("Battery SOC")

plt.figure()
plt.plot(battery_power)
plt.xlabel("Time (min)")
plt.ylabel("Battery power")

plt.figure()
plt.plot(balance)
plt.xlabel("Time (min)")
plt.ylabel("Balance")

plt.figure()
plt.plot(genset_0_power)
plt.plot(genset_1_power)
plt.plot(genset_2_power)
plt.plot(genset_power)
plt.xlabel("Time (min)")
plt.ylabel("Genset power")
plt.legend(["Genset 0", "Genset 1", "Genset 2", "Genset group"])

plt.figure()
plt.plot(genset_0_overload)
plt.plot(genset_1_overload)
plt.plot(genset_2_overload)
plt.xlabel("Time (min)")
plt.ylabel("Genset overload")
plt.legend(["Genset 0", "Genset 1", "Genset 2"])

plt.figure()
plt.plot(genset_0_avgpower)
plt.plot(genset_1_avgpower)
plt.plot(genset_2_avgpower)
plt.xlabel("Time (min)")
plt.ylabel("Genset average power on the last 48 running hours")
plt.legend(["Genset 0", "Genset 1", "Genset 2"])

plt.figure()
plt.plot(demand)
plt.xlabel("Time (min)")
plt.ylabel("Demand")

plt.figure()
plt.plot(number_genset_running)
plt.xlabel("Time (min)")
plt.ylabel("Number of running gensets")

plt.figure()
plt.plot(number_genset_warmup)
plt.xlabel("Time (min)")
plt.ylabel("Number of warmup gensets")


assert np.abs(observations['device_observations']['genset_group']['device_observations']['genset_group_active_power'] - observations['device_observations']['demand']['device_observations']['demand']) < 10e-2
assert np.abs(observations['device_observations']['battery']['device_observations']['p_grid']) < 10e-2
assert observations['device_observations']['balance'] == 0
assert observations['device_observations']['genset_group']['device_observations']['config_lists']['running_gensets_ids'] == [0,1]
assert np.all(np.array(demand) >= -1e-6)
assert np.all(np.abs(np.array(balance)) <= 1e-6)
assert np.all(np.array(genset_0_overload) == 0)
assert np.all(np.array(genset_1_overload) == 0)
assert np.all(np.array(genset_2_overload) == 0)


Run a bit longer. The demand will increase. When this happens, the gensets will increase their average running power and will need to be limited. That's where the third generator is started. 

In [ ]:
import matplotlib.pyplot as plt

action = {
    "genset_group": {"status_change": "none"},
    "battery": {"p_grid": 0}
}

for i in range(250):
    safe_action = microgrid_controller.generate_safe_action(action)
    microgrid.step(safe_action)
    microgrid_controller.update_controller_state({'device_observations': microgrid.gather_observations(), 'controller_state': {}}) 
    observations = microgrid_controller.gather_observations()  
    battery_soc.append(observations['device_observations']['battery']['device_observations']['soc'])
    battery_power.append(observations['device_observations']['battery']['device_observations']['p_grid'])
    balance.append(observations['device_observations']['balance'])
    genset_power.append(observations['device_observations']['genset_group']['device_observations']['genset_group_active_power'])
    genset_0_power.append(observations['device_observations']['genset_group']['device_observations']['gensets'][0]['device_observations']['active_power'])
    genset_1_power.append(observations['device_observations']['genset_group']['device_observations']['gensets'][1]['device_observations']['active_power'])
    genset_2_power.append(observations['device_observations']['genset_group']['device_observations']['gensets'][2]['device_observations']['active_power'])
    genset_0_overload.append(1 if observations['device_observations']['genset_group']['device_observations']['gensets'][0]['device_observations']['overload'] else 0)
    genset_1_overload.append(1 if observations['device_observations']['genset_group']['device_observations']['gensets'][1]['device_observations']['overload'] else 0)
    genset_2_overload.append(1 if observations['device_observations']['genset_group']['device_observations']['gensets'][2]['device_observations']['overload'] else 0)
    genset_0_avgpower.append(observations['device_observations']['genset_group']['device_observations']['gensets'][0]['controller_state']['average_active_power'])
    genset_1_avgpower.append(observations['device_observations']['genset_group']['device_observations']['gensets'][1]['controller_state']['average_active_power'])
    genset_2_avgpower.append(observations['device_observations']['genset_group']['device_observations']['gensets'][2]['controller_state']['average_active_power'])
    demand.append(observations['device_observations']['demand']['device_observations']['demand'])
    number_genset_running.append(len(observations['device_observations']['genset_group']['device_observations']['config_lists']['running_gensets_ids']))
    number_genset_warmup.append(len(observations['device_observations']['genset_group']['device_observations']['config_lists']['warmup_gensets_ids']))



plt.plot(battery_soc)
plt.xlabel("Time (min)")
plt.ylabel("Battery SOC")

plt.figure()
plt.plot(battery_power)
plt.xlabel("Time (min)")
plt.ylabel("Battery power")

plt.figure()
plt.plot(balance)
plt.xlabel("Time (min)")
plt.ylabel("Balance")

plt.figure()
plt.plot(genset_0_power)
plt.plot(genset_1_power)
plt.plot(genset_2_power)
plt.plot(genset_power)
plt.xlabel("Time (min)")
plt.ylabel("Genset power")
plt.legend(["Genset 0", "Genset 1", "Genset 2", "Genset group"])

plt.figure()
plt.plot(genset_0_avgpower)
plt.plot(genset_1_avgpower)
plt.plot(genset_2_avgpower)
plt.xlabel("Time (min)")
plt.ylabel("Genset average power on the last 48 running hours")
plt.legend(["Genset 0", "Genset 1", "Genset 2"])


plt.figure()
plt.plot(demand)
plt.xlabel("Time (min)")
plt.ylabel("Demand")

plt.figure()
plt.plot(number_genset_running)
plt.xlabel("Time (min)")
plt.ylabel("Number of running gensets")

plt.figure()
plt.plot(number_genset_warmup)
plt.xlabel("Time (min)")
plt.ylabel("Number of warmup gensets")


assert np.abs(observations['device_observations']['genset_group']['device_observations']['genset_group_active_power'] - observations['device_observations']['demand']['device_observations']['demand']) < 10e-2
assert np.abs(observations['device_observations']['battery']['device_observations']['p_grid']) < 10e-2
assert observations['device_observations']['balance'] == 0
assert observations['device_observations']['genset_group']['device_observations']['config_lists']['running_gensets_ids'] == [0,1,2]
assert np.all(np.array(demand) >= -1e-6)
assert np.all(np.abs(np.array(balance)) <= 1e-6)

# The gensets are not overloaded, and the average power is below 0.7 * prime_power_rating
assert np.all(np.array(genset_0_overload) == 0)
assert np.all(np.array(genset_1_overload) == 0)
assert np.all(np.array(genset_2_overload) == 0)

assert np.all(np.array(genset_0_avgpower) <= 280)
assert np.all(np.array(genset_1_avgpower) <= 280)
assert np.all(np.array(genset_2_avgpower) <= 280)


Now we will charge the battery as much as possible and always ask to turn off the generators. The shield should prevent it from happening until it is safe, then it will be done, and once the battery is discharged completely the generators will be turned on again - without any negative demand. This will happen several times in a row.

Note: we restart the plots from here to see better, but we start from last experiment.

In [ ]:
action = {
    "genset_group": {"status_change": "stop_last"},
    "battery": {"p_grid": -np.inf}
}


genset_power = []
battery_soc = []
battery_power = []
balance = []
genset_0_power = []
genset_1_power = []
genset_2_power = []
genset_0_avgpower = []
genset_1_avgpower = []
genset_2_avgpower = []
demand = []
number_genset_running = []
number_genset_warmup = []
number_genset_cooldown = []



for i in range(150):
    safe_action = microgrid_controller.generate_safe_action(action)
    microgrid.step(safe_action)
    microgrid_controller.update_controller_state({'device_observations': microgrid.gather_observations(), 'controller_state': {}}) 
    observations = microgrid_controller.gather_observations()  
    battery_soc.append(observations['device_observations']['battery']['device_observations']['soc'])
    battery_power.append(observations['device_observations']['battery']['device_observations']['p_grid'])
    balance.append(observations['device_observations']['balance'])
    genset_power.append(observations['device_observations']['genset_group']['device_observations']['genset_group_active_power'])
    genset_0_power.append(observations['device_observations']['genset_group']['device_observations']['gensets'][0]['device_observations']['active_power'])
    genset_1_power.append(observations['device_observations']['genset_group']['device_observations']['gensets'][1]['device_observations']['active_power'])
    genset_2_power.append(observations['device_observations']['genset_group']['device_observations']['gensets'][2]['device_observations']['active_power'])
    genset_0_overload.append(1 if observations['device_observations']['genset_group']['device_observations']['gensets'][0]['device_observations']['overload'] else 0)
    genset_1_overload.append(1 if observations['device_observations']['genset_group']['device_observations']['gensets'][1]['device_observations']['overload'] else 0)
    genset_2_overload.append(1 if observations['device_observations']['genset_group']['device_observations']['gensets'][2]['device_observations']['overload'] else 0)
    genset_0_avgpower.append(observations['device_observations']['genset_group']['device_observations']['gensets'][0]['controller_state']['average_active_power'])
    genset_1_avgpower.append(observations['device_observations']['genset_group']['device_observations']['gensets'][1]['controller_state']['average_active_power'])
    genset_2_avgpower.append(observations['device_observations']['genset_group']['device_observations']['gensets'][2]['controller_state']['average_active_power'])
    demand.append(observations['device_observations']['demand']['device_observations']['demand'])
    number_genset_running.append(len(observations['device_observations']['genset_group']['device_observations']['config_lists']['running_gensets_ids']))
    number_genset_warmup.append(len(observations['device_observations']['genset_group']['device_observations']['config_lists']['warmup_gensets_ids']))
    number_genset_cooldown.append(len(observations['device_observations']['genset_group']['device_observations']['config_lists']['cooldown_gensets_ids']))


plt.plot(battery_soc)
plt.xlabel("Time (min)")
plt.ylabel("Battery SOC")

plt.figure()
plt.plot(battery_power)
plt.xlabel("Time (min)")
plt.ylabel("Battery power")

plt.figure()
plt.plot(balance)
plt.xlabel("Time (min)")
plt.ylabel("Balance")

plt.figure()
plt.plot(genset_0_power)
plt.plot(genset_1_power)
plt.plot(genset_2_power)
plt.plot(genset_power)
plt.xlabel("Time (min)")
plt.ylabel("Genset power")
plt.legend(["Genset 0", "Genset 1", "Genset 2", "Genset group"])

plt.figure()
plt.plot(genset_0_avgpower)
plt.plot(genset_1_avgpower)
plt.plot(genset_2_avgpower)
plt.xlabel("Time (min)")
plt.ylabel("Genset average power on the last 48 running hours")
plt.legend(["Genset 0", "Genset 1", "Genset 2"])


plt.figure()
plt.plot(demand)
plt.xlabel("Time (min)")
plt.ylabel("Demand")

plt.figure()
plt.plot(number_genset_running)
plt.xlabel("Time (min)")
plt.ylabel("Number of running gensets")

plt.figure()
plt.plot(number_genset_warmup)
plt.xlabel("Time (min)")
plt.ylabel("Number of warmup gensets")

plt.figure()
plt.plot(number_genset_cooldown)
plt.xlabel("Time (min)")
plt.ylabel("Number of cooldown gensets")



assert np.all(np.abs(np.array(balance)) <= 1e-6)


# Microgrid with turbine

## Wind turbine dynamics

In [ ]:
from microgrid import MicroGrid
from microgrid_controller import MicroGridController
import datetime
import numpy as np
import pathlib

In [ ]:
external_world_params = {
    "init_date_time": {"value": datetime.datetime(2018, 6, 6, 6, 6, 0), "type": "datetime"},
    "time_step": {"value": 1, "type": "integer"}
}

genset_params = {
    'device': {
        'const_params': {
            'alpha_g': {'value': 0.25, 'type': 'float'}, 'beta_g': {'value': 10, 'type': 'float'},
            'prime_power_rating': {'value': 400, 'type': 'float'}, 'temp_overload_factor': {'value': 1.1, 'type': 'float'},
            'time_step': {'value': 1, 'type': 'int'}, 'noise_level': {'value': 'none', 'type': 'string'},
            'noise_range_percentage': {'value': 0.2, 'type': 'float'}
        },
        'init_params': {
            'running': {'value': False, 'type': 'bool'}, 'init_power_setpoint': {'value': 300, 'type': 'float'}
        }
    },
    'controller': {
        'const_params': {
            'minimum_run_time': {'value': 30, 'type': 'integer'}, 'minimum_load': {'value': 120, 'type': 'float'},
            'warmup_power': {'value': 100, 'type': 'float'}, 'cooldown_power': {'value': 0, 'type': 'float'},
            'warmup_time': {'value': 3, 'type': 'integer'}, 'cooldown_time': {'value': 6, 'type': 'integer'},
            'average_active_power_constraint': {'value': True, 'type': 'boolean'},
            'average_active_power_window': {'value': 2880, 'type': 'integer'},
            'average_active_power_limit': {'value': 0.7, 'type': 'float'}
        },
        'init_params': {
            'status': {'value': 'off', 'type': 'string', 'elements': ['off', 'running', 'warmup', 'cooldown']},
            'time_since_warmup': {'value': 0, 'type': 'integer'}, 'time_since_cooldown': {'value': 0, 'type': 'integer'},
            'time_since_start': {'value': 0, 'type': 'integer'}, 'average_active_power': {'value': 0, 'type': 'float'}
        }
    }
}

genset_group_params = {
    'device': {
        'const_params': {'n_gensets': {'value': 3, 'type': 'integer'}},
        'init_params': {'gensets': [genset_params] * 3, 'obs_type': 'params'}
    },
    'controller': {
        'const_params': {'priority_order': {'value': True, 'type': 'boolean'}},
        'init_params': {'obs_type': 'params'}
    }
}

battery_params = {
    "device": {
        "const_params": {
            "P_nom": {"value": 600, "type": "float"},
            "soc_min": {"value": 0, "type": "float"},
            "soc_max": {"value": 1, "type": "float"},
            "I_nom": {"value": 672, "type": "float"},
            "Q_max": {"value": 672, "type": "float"},
            "eta_charge": {"value": 0.95, "type": "float"},
            "R": {"value": 0.039375, "type": "float"},
            "A": {"value": 39.6, "type": "float"},
            "B": {"value": 929.16, "type": "float"},
            "alpha_d": {"value": 5, "type": "float"},
            "beta": {"value": 1, "type": "float"},
            "degradation_cost_type": {"value": "cycle_based", "type": "string"},
            "buffer_size": {"value": 30, "type": "integer"},
            "noise_level": {"value": "none", "type": "string"},
            "noise_range_percentage": {"value": 0.2, "type": "float"},
        },
        "init_params": {
            "soc": {"value": 0.5, "type": "float", "min": 0.1, "max": 0.9},
            "p_grid": {"value": 0, "type": "float"},
            "obs_type": 'params'
        },
    },
    "controller": {
        "const_params": {
            "soc_max_norm": {"value": 0.9, "type": "float"},
            "soc_min_norm": {"value": 0.1, "type": "float"},
            "soc_max_res": {"value": 0.95, "type": "float"},
            "soc_min_res": {"value": 0.05, "type": "float"},
        },
        "init_params": {
            "obs_type": 'params'
        },
    },
}


demand_params = {
    'controller': {'const_params':{}, 'init_params':{'obs_type': 'params'}},
    'device': {
        'const_params': {
            "mode": {'value': 'data', 'type': 'string'},
            "pred_time_step": {'value': 1, 'type': 'integer'},
            "nb_pred_time_steps": {'value': 10, 'type': 'integer'},
            "normalisation_factor": {'value': 300, 'type': 'integer'},       # Back to multiply demand by 1        
            "forecast_model": {'value': 'forecast', 'type': 'string'},
        },
        'init_params': {
            "obs_type": 'params',
        }
    }
}


wind_turbine_params = {
    'device': {
        'init_params': {
            'turbine_setpoint': {'value': 300, 'type': 'float'},
            'active': {'value': True, 'type': 'boolean'},
            'obs_type': 'params'
        },
        'const_params': {
            'pred_time_step': {'value': 1, 'type': 'integer'},
            'nb_pred_time_steps': {'value': 10, 'type': 'integer'},
            'mode': {'value': 'perlin', 'type': 'string'},
            'nominal_power': {'value': 500, 'type': 'float'},
            'perlin_params': {
                'average': {'value': 200, 'type': 'float'},
                'nb_octaves': {'value': 5, 'type': 'integer'},
                'octaves_step': {'value': 4, 'type': 'integer'},
                'period': {'value': 60*24, 'type': 'integer'},
                'persistence': {'value': 0.8, 'type': 'float'},
                'lacunarity': {'value': 2.5, 'type': 'float'},
                'repeat': {'value': 1024, 'type': 'integer'}
            }
        }
    },
    'controller': {
        'init_params': {
            'turbine_setpoint': {'value': 0, 'type': 'float'},
            'obs_type': 'params'
        },
        'const_params': {
            'power_max': {'value': 500, 'type': 'float'},
            'power_min': {'value': 0, 'type': 'float'},
        }
    }
}



microgrid_params = {
    "device": {
        "const_params": {
        },
        "init_params": {
            "external_world": external_world_params,
            "genset_group": genset_group_params,
            "battery": battery_params,
            "demand": demand_params,
            "wind_turbine": wind_turbine_params,
            "obs_type": 'params'
        },
    },
    "controller": {
        "const_params": {
            "check_initialization": {"value": True, "type": "boolean"},
            "init_check_steps": {"value": 8, "type": "integer"},
            "init_check_loops": {"value": 3, "type": "integer"},
            "wind_turbine_priority": {"value": True, "type": "boolean"},
            "check_balance": {"value": True, "type": "boolean"},                       #  Check balance is ON --> Predictive shield on generator status if steady demand in next 10 time step is irrecoverable without reserve
            "check_reserve": {"value": False, "type": "boolean"},                       # Still no check reserve --> No predictive shield on generator status if worst case demand in next 10 time step is irrecoverable with reserve
            "adjust_battery": {"value": True, "type": "boolean"},
            "conservativeness_coeff": {"value": 1, "type": "float"},
            "min_available_wind_power": {"value": 0, "type": "float"},
            "max_available_wind_power_drop_step": {"value": 0.18, "type": "float"},
            "max_demand": {"value": 570, "type": "float"},
            "max_demand_increase_step": {"value": 80, "type": "float"},
        },
        "init_params": {
            "obs_type": 'params',
        },
    }
}

time_step = 1


In [ ]:
microgrid = MicroGrid(microgrid_params['device'], time_step, real=True)
microgrid_controller = MicroGridController(microgrid_params, time_step)
microgrid_controller.update_controller_state({'device_observations': microgrid.gather_observations(), 'controller_state': {}})  # Initialize the controller with the current state of the microgrid  


There is a lot of wind. Without the help of the battery, the wind turbine is able to cover for the necessary power.

In [ ]:
action = {
    "genset_group": {"status_change": "none"},
    "battery": {"p_grid": 0},
}
safe_action = microgrid_controller.generate_safe_action(action)
microgrid.step(safe_action)
microgrid_controller.update_controller_state({'device_observations': microgrid.gather_observations(), 'controller_state': {}}) 
observations = microgrid_controller.gather_observations()  

assert observations['device_observations']['genset_group']['device_observations']['genset_group_active_power'] == 0
assert np.abs(observations['device_observations']['battery']['device_observations']['p_grid']) < 10e-2
assert np.abs(observations['device_observations']['wind_turbine']['device_observations']['wind_power'] - observations['device_observations']['demand']['device_observations']['demand']) < 10e-2
assert observations['device_observations']['balance'] == 0



In [ ]:
observations['device_observations']['wind_turbine']['device_observations']['wind_power']

In [ ]:
observations['device_observations']['demand']['device_observations']['demand']

In [ ]:
observations

We now want to charge the battery a bit. The wind turbine will cover it too.

In [ ]:
action = {
    "genset_group": {"status_change": "none"},
    "battery": {"p_grid": -50},
}
safe_action = microgrid_controller.generate_safe_action(action)
microgrid.step(safe_action)
microgrid_controller.update_controller_state({'device_observations': microgrid.gather_observations(), 'controller_state': {}}) 
observations = microgrid_controller.gather_observations()  

assert observations['device_observations']['genset_group']['device_observations']['genset_group_active_power'] == 0
assert np.round(observations['device_observations']['battery']['device_observations']['p_grid']) == -50
assert np.abs(observations['device_observations']['wind_turbine']['device_observations']['wind_power'] - observations['device_observations']['demand']['device_observations']['demand'] - 50) < 10e-2
assert observations['device_observations']['balance'] == 0


Now if the battery asks for too much, the power it actually receive will be reduced by the shield so that the demand is supplied first.

In [ ]:
action = {
    "genset_group": {"status_change": "none"},
    "battery": {"p_grid": -5000},
}

safe_action = microgrid_controller.generate_safe_action(action)
microgrid.step(safe_action)
microgrid_controller.update_controller_state({'device_observations': microgrid.gather_observations(), 'controller_state': {}}) 
observations = microgrid_controller.gather_observations()  

assert observations['device_observations']['genset_group']['device_observations']['genset_group_active_power'] == 0
assert observations['device_observations']['battery']['device_observations']['p_grid'] == observations['device_observations']['demand']['device_observations']['demand'] - observations['device_observations']['wind_turbine']['device_observations']['available_wind_power']
assert observations['device_observations']['wind_turbine']['device_observations']['wind_power'] == observations['device_observations']['wind_turbine']['device_observations']['available_wind_power']
assert observations['device_observations']['balance'] == 0


Let start a generator, with the battery idle. The warmup power of the generator should be considered in the wind turbine setpoint.

In [ ]:
action = {
    "genset_group": {"status_change": "start_next"},
    "battery": {"p_grid": 0},
}

safe_action = microgrid_controller.generate_safe_action(action)
microgrid.step(safe_action)
microgrid_controller.update_controller_state({'device_observations': microgrid.gather_observations(), 'controller_state': {}}) 
observations = microgrid_controller.gather_observations()  

assert observations['device_observations']['genset_group']['device_observations']['genset_group_active_power'] == genset_params['controller']['const_params']['warmup_power']['value']
assert observations['device_observations']['battery']['device_observations']['p_grid'] == 0
assert observations['device_observations']['wind_turbine']['device_observations']['wind_power'] == observations['device_observations']['demand']['device_observations']['demand'] - genset_params['controller']['const_params']['warmup_power']['value']
assert observations['device_observations']['balance'] == 0


Pass the warmup. The generator setpoint should be set to minimal because the turbine takes everything except the minimal power from the generator.

In [ ]:
# Pass the warmup
action = {
    "genset_group": {"status_change": "none"},
    "battery": {"p_grid": 0},
}

for i in range(2):      # Pass the warmup
    safe_action = microgrid_controller.generate_safe_action(action)
    microgrid.step(safe_action)
    microgrid_controller.update_controller_state({'device_observations': microgrid.gather_observations(), 'controller_state': {}}) 
    observations = microgrid_controller.gather_observations()  
    assert observations['device_observations']['genset_group']['device_observations']['genset_group_active_power'] == genset_params['controller']['const_params']['warmup_power']['value']
    assert observations['device_observations']['battery']['device_observations']['p_grid'] == 0
    assert observations['device_observations']['wind_turbine']['device_observations']['wind_power'] == observations['device_observations']['demand']['device_observations']['demand'] - genset_params['controller']['const_params']['warmup_power']['value']
    assert observations['device_observations']['balance'] == 0


# Run with one generator ON
safe_action = microgrid_controller.generate_safe_action(action)
microgrid.step(safe_action)
microgrid_controller.update_controller_state({'device_observations': microgrid.gather_observations(), 'controller_state': {}}) 
observations = microgrid_controller.gather_observations()  

assert observations['device_observations']['genset_group']['device_observations']['genset_group_active_power'] == genset_params['controller']['const_params']['minimum_load']['value']
assert observations['device_observations']['battery']['device_observations']['p_grid'] == 0
assert observations['device_observations']['wind_turbine']['device_observations']['wind_power'] == observations['device_observations']['demand']['device_observations']['demand'] - genset_params['controller']['const_params']['minimum_load']['value']
assert observations['device_observations']['balance'] == 0


### Double demand

Start again - double the demand so that the wind turbine does not cover



In [ ]:
from microgrid import MicroGrid
from microgrid_controller import MicroGridController
import datetime
import numpy as np
import pathlib

In [ ]:
external_world_params = {
    "init_date_time": {"value": datetime.datetime(2018, 6, 6, 6, 6, 0), "type": "datetime"},
    "time_step": {"value": 1, "type": "integer"}
}

genset_params = {
    'device': {
        'const_params': {
            'alpha_g': {'value': 0.25, 'type': 'float'}, 'beta_g': {'value': 10, 'type': 'float'},
            'prime_power_rating': {'value': 400, 'type': 'float'}, 'temp_overload_factor': {'value': 1.1, 'type': 'float'},
            'time_step': {'value': 1, 'type': 'int'}, 'noise_level': {'value': 'none', 'type': 'string'},
            'noise_range_percentage': {'value': 0.2, 'type': 'float'}
        },
        'init_params': {
            'running': {'value': False, 'type': 'bool'}, 'init_power_setpoint': {'value': 300, 'type': 'float'}
        }
    },
    'controller': {
        'const_params': {
            'minimum_run_time': {'value': 30, 'type': 'integer'}, 'minimum_load': {'value': 120, 'type': 'float'},
            'warmup_power': {'value': 100, 'type': 'float'}, 'cooldown_power': {'value': 0, 'type': 'float'},
            'warmup_time': {'value': 3, 'type': 'integer'}, 'cooldown_time': {'value': 6, 'type': 'integer'},
            'average_active_power_constraint': {'value': True, 'type': 'boolean'},
            'average_active_power_window': {'value': 2880, 'type': 'integer'},
            'average_active_power_limit': {'value': 0.7, 'type': 'float'}
        },
        'init_params': {
            'status': {'value': 'off', 'type': 'string', 'elements': ['off', 'running', 'warmup', 'cooldown']},
            'time_since_warmup': {'value': 0, 'type': 'integer'}, 'time_since_cooldown': {'value': 0, 'type': 'integer'},
            'time_since_start': {'value': 0, 'type': 'integer'}, 'average_active_power': {'value': 0, 'type': 'float'}
        }
    }
}

genset_group_params = {
    'device': {
        'const_params': {'n_gensets': {'value': 3, 'type': 'integer'}},
        'init_params': {'gensets': [genset_params] * 3, 'obs_type': 'params'}
    },
    'controller': {
        'const_params': {'priority_order': {'value': True, 'type': 'boolean'}},
        'init_params': {'obs_type': 'params'}
    }
}

battery_params = {
    "device": {
        "const_params": {
            "P_nom": {"value": 600, "type": "float"},
            "soc_min": {"value": 0, "type": "float"},
            "soc_max": {"value": 1, "type": "float"},
            "I_nom": {"value": 672, "type": "float"},
            "Q_max": {"value": 672, "type": "float"},
            "eta_charge": {"value": 0.95, "type": "float"},
            "R": {"value": 0.039375, "type": "float"},
            "A": {"value": 39.6, "type": "float"},
            "B": {"value": 929.16, "type": "float"},
            "alpha_d": {"value": 5, "type": "float"},
            "beta": {"value": 1, "type": "float"},
            "degradation_cost_type": {"value": "cycle_based", "type": "string"},
            "buffer_size": {"value": 30, "type": "integer"},
            "noise_level": {"value": "none", "type": "string"},
            "noise_range_percentage": {"value": 0.2, "type": "float"},
        },
        "init_params": {
            "soc": {"value": 0.5, "type": "float", "min": 0.1, "max": 0.9},
            "p_grid": {"value": 0, "type": "float"},
            "obs_type": 'params'
        },
    },
    "controller": {
        "const_params": {
            "soc_max_norm": {"value": 0.9, "type": "float"},
            "soc_min_norm": {"value": 0.1, "type": "float"},
            "soc_max_res": {"value": 0.95, "type": "float"},
            "soc_min_res": {"value": 0.05, "type": "float"},
        },
        "init_params": {
            "obs_type": 'params'
        },
    },
}


demand_params = {
    'controller': {'const_params':{}, 'init_params':{'obs_type': 'params'}},
    'device': {
        'const_params': {
            "mode": {'value': 'data', 'type': 'string'},
            "pred_time_step": {'value': 1, 'type': 'integer'},
            "nb_pred_time_steps": {'value': 10, 'type': 'integer'},
            "normalisation_factor": {'value': 600, 'type': 'integer'},       # Multiply by 2       
            "forecast_model": {'value': 'forecast', 'type': 'string'},
        },
        'init_params': {
            "obs_type": 'params',
        }
    }
}


wind_turbine_params = {
    'device': {
        'init_params': {
            'turbine_setpoint': {'value': 300, 'type': 'float'},
            'active': {'value': True, 'type': 'boolean'},
            'obs_type': 'params'
        },
        'const_params': {
            'pred_time_step': {'value': 1, 'type': 'integer'},
            'nb_pred_time_steps': {'value': 10, 'type': 'integer'},
            'mode': {'value': 'perlin', 'type': 'string'},
            'nominal_power': {'value': 500, 'type': 'float'},
            'perlin_params': {
                'average': {'value': 200, 'type': 'float'},
                'nb_octaves': {'value': 5, 'type': 'integer'},
                'octaves_step': {'value': 4, 'type': 'integer'},
                'period': {'value': 60*24, 'type': 'integer'},
                'persistence': {'value': 0.8, 'type': 'float'},
                'lacunarity': {'value': 2.5, 'type': 'float'},
                'repeat': {'value': 1024, 'type': 'integer'}
            }
        }
    },
    'controller': {
        'init_params': {
            'turbine_setpoint': {'value': 0, 'type': 'float'},
            'obs_type': 'params'
        },
        'const_params': {
            'power_max': {'value': 500, 'type': 'float'},
            'power_min': {'value': 0, 'type': 'float'},
        }
    }
}



microgrid_params = {
    "device": {
        "const_params": {
        },
        "init_params": {
            "external_world": external_world_params,
            "genset_group": genset_group_params,
            "battery": battery_params,
            "demand": demand_params,
            "wind_turbine": wind_turbine_params,
            "obs_type": 'params'
        },
    },
    "controller": {
        "const_params": {
            "check_initialization": {"value": True, "type": "boolean"},
            "init_check_steps": {"value": 8, "type": "integer"},
            "init_check_loops": {"value": 3, "type": "integer"},
            "wind_turbine_priority": {"value": False, "type": "boolean"},               # Wind turbine priority is OFF --> The battery can be asked to discharge even if the wind turbine would be able to cover the demand
            "check_balance": {"value": True, "type": "boolean"},                       #  Check balance is ON --> Predictive shield on generator status if steady demand in next 10 time step is irrecoverable without reserve
            "check_reserve": {"value": False, "type": "boolean"},                       # Still no check reserve --> No predictive shield on generator status if worst case demand in next 10 time step is irrecoverable with reserve
            "adjust_battery": {"value": True, "type": "boolean"},
            "conservativeness_coeff": {"value": 1, "type": "float"},
            "min_available_wind_power": {"value": 0, "type": "float"},
            "max_available_wind_power_drop_step": {"value": 0.18, "type": "float"},
            "max_demand": {"value": 570, "type": "float"},
            "max_demand_increase_step": {"value": 80, "type": "float"},
        },
        "init_params": {
            "obs_type": 'params',
        },
    }
}

time_step = 1


microgrid = MicroGrid(microgrid_params['device'], time_step, real=True)
microgrid_controller = MicroGridController(microgrid_params, time_step)
# Initialize the controller with the current state of the microgrid
microgrid_controller.update_controller_state({'device_observations': microgrid.gather_observations(), 'controller_state': {}}) 


Make one step. The turbine should give all it can, and the battery should cover the rest.

In [ ]:
action = {
    "genset_group": {"status_change": "none"},
    "battery": {"p_grid": 0},
}

safe_action = microgrid_controller.generate_safe_action(action)
microgrid.step(safe_action)
microgrid_controller.update_controller_state({'device_observations': microgrid.gather_observations(), 'controller_state': {}}) 
observations = microgrid_controller.gather_observations()  

assert observations['device_observations']['genset_group']['device_observations']['genset_group_active_power'] == 0
assert observations['device_observations']['battery']['device_observations']['p_grid'] == observations['device_observations']['demand']['device_observations']['demand'] - observations['device_observations']['wind_turbine']['device_observations']['available_wind_power']
assert observations['device_observations']['wind_turbine']['device_observations']['wind_power'] == observations['device_observations']['wind_turbine']['device_observations']['available_wind_power']
assert observations['device_observations']['balance'] == 0



Turn the generator on. The battery will discharge less.

In [ ]:
action = {
    "genset_group": {"status_change": "start_next"},
    "battery": {"p_grid": 0},
}

safe_action = microgrid_controller.generate_safe_action(action)
microgrid.step(safe_action)
microgrid_controller.update_controller_state({'device_observations': microgrid.gather_observations(), 'controller_state': {}}) 
observations = microgrid_controller.gather_observations()  

assert observations['device_observations']['genset_group']['device_observations']['genset_group_active_power'] == genset_params['controller']['const_params']['warmup_power']['value']
assert observations['device_observations']['battery']['device_observations']['p_grid'] == observations['device_observations']['demand']['device_observations']['demand'] - observations['device_observations']['wind_turbine']['device_observations']['available_wind_power'] - genset_params['controller']['const_params']['warmup_power']['value']
assert observations['device_observations']['wind_turbine']['device_observations']['wind_power'] == observations['device_observations']['wind_turbine']['device_observations']['available_wind_power']   # Used to the max
assert observations['device_observations']['balance'] == 0


Wait until warmup is over. With 0 as the battery power command, the wind turbine gives all it can, the genset handles the difference.

In [ ]:
action = {
    "genset_group": {"status_change": "none"},
    "battery": {"p_grid": 0},
}

for i in range(2):    # Pass the warmup
    safe_action = microgrid_controller.generate_safe_action(action)
    microgrid.step(safe_action)
    microgrid_controller.update_controller_state({'device_observations': microgrid.gather_observations(), 'controller_state': {}}) 
    observations = microgrid_controller.gather_observations()  

    assert observations['device_observations']['genset_group']['device_observations']['genset_group_active_power'] == genset_params['controller']['const_params']['warmup_power']['value']
    assert observations['device_observations']['battery']['device_observations']['p_grid'] == observations['device_observations']['demand']['device_observations']['demand'] - observations['device_observations']['wind_turbine']['device_observations']['available_wind_power'] - genset_params['controller']['const_params']['warmup_power']['value']
    assert observations['device_observations']['wind_turbine']['device_observations']['wind_power'] == observations['device_observations']['wind_turbine']['device_observations']['available_wind_power']   # Used to the max
    assert observations['device_observations']['balance'] == 0

safe_action = microgrid_controller.generate_safe_action(action)
microgrid.step(safe_action)
microgrid_controller.update_controller_state({'device_observations': microgrid.gather_observations(), 'controller_state': {}}) 
observations = microgrid_controller.gather_observations()

assert np.abs(observations['device_observations']['genset_group']['device_observations']['genset_group_active_power'] - (observations['device_observations']['demand']['device_observations']['demand'] - observations['device_observations']['wind_turbine']['device_observations']['available_wind_power'] )) < 10e-2
assert np.abs(observations['device_observations']['battery']['device_observations']['p_grid']) < 10e-2
assert observations['device_observations']['wind_turbine']['device_observations']['wind_power'] == observations['device_observations']['wind_turbine']['device_observations']['available_wind_power']   # Used to the max
assert observations['device_observations']['balance'] == 0

Start charging the battery. The genset will provide more.



In [ ]:
action = {
    "genset_group": {"status_change": "none"},
    "battery": {"p_grid": -50},
}

safe_action = microgrid_controller.generate_safe_action(action)
microgrid.step(safe_action)
microgrid_controller.update_controller_state({'device_observations': microgrid.gather_observations(), 'controller_state': {}}) 
observations = microgrid_controller.gather_observations()


expected_genset_group_active_power = observations['device_observations']['demand']['device_observations']['demand'] - observations['device_observations']['wind_turbine']['device_observations']['available_wind_power'] - action['battery']['p_grid']
expected_battery_p_grid = action['battery']['p_grid']
expected_wind_power = observations['device_observations']['wind_turbine']['device_observations']['available_wind_power']
expected_balance = 0

assert np.abs(observations['device_observations']['genset_group']['device_observations']['genset_group_active_power'] - expected_genset_group_active_power) < 10e-2
assert np.abs(observations['device_observations']['battery']['device_observations']['p_grid'] - expected_battery_p_grid) < 10e-2
assert np.abs(observations['device_observations']['wind_turbine']['device_observations']['wind_power'] - expected_wind_power)< 10e-2
assert np.abs(observations['device_observations']['balance'] - expected_balance)< 10e-2

Make the battery ask too much. It is corrected given the genset and wind turbine to ensure good balance.



In [ ]:
action = {
    "genset_group": {"status_change": "none"},
    "battery": {"p_grid": -600},
}

safe_action = microgrid_controller.generate_safe_action(action)
microgrid.step(safe_action)
microgrid_controller.update_controller_state({'device_observations': microgrid.gather_observations(), 'controller_state': {}}) 
observations = microgrid_controller.gather_observations()

expected_genset_group_active_power = genset_params['device']['const_params']['prime_power_rating']['value'] # Maximum
expected_battery_p_grid = observations['device_observations']['demand']['device_observations']['demand'] - observations['device_observations']['wind_turbine']['device_observations']['available_wind_power'] - genset_params['device']['const_params']['prime_power_rating']['value']
expected_wind_power = observations['device_observations']['wind_turbine']['device_observations']['available_wind_power']  # Maximum
expected_balance = 0

assert np.abs(observations['device_observations']['genset_group']['device_observations']['genset_group_active_power'] - expected_genset_group_active_power) < 10e-2
assert np.abs(observations['device_observations']['battery']['device_observations']['p_grid'] - expected_battery_p_grid) < 10e-2
assert np.abs(observations['device_observations']['wind_turbine']['device_observations']['wind_power'] - expected_wind_power)< 10e-2
assert np.abs(observations['device_observations']['balance'] - expected_balance)< 10e-2

Make the battery discharge. The generator will first reduce its contribution.

In [ ]:
action = {
    "genset_group": {"status_change": "none"},
    "battery": {"p_grid": 10},
}

safe_action = microgrid_controller.generate_safe_action(action)
microgrid.step(safe_action)
microgrid_controller.update_controller_state({'device_observations': microgrid.gather_observations(), 'controller_state': {}}) 
observations = microgrid_controller.gather_observations()

expected_genset_group_active_power = observations['device_observations']['demand']['device_observations']['demand']  - observations['device_observations']['wind_turbine']['device_observations']['available_wind_power'] - action['battery']['p_grid']
expected_battery_p_grid = action['battery']['p_grid']
expected_wind_power = observations['device_observations']['wind_turbine']['device_observations']['available_wind_power']  # Maximum
expected_balance = 0

assert np.abs(observations['device_observations']['genset_group']['device_observations']['genset_group_active_power'] - expected_genset_group_active_power) < 10e-2
assert np.abs(observations['device_observations']['battery']['device_observations']['p_grid'] - expected_battery_p_grid) < 10e-2
assert np.abs(observations['device_observations']['wind_turbine']['device_observations']['wind_power'] - expected_wind_power)< 10e-2
assert np.abs(observations['device_observations']['balance'] - expected_balance)< 10e-2

Make the battery discharge more. The generator will reduce at minimum load. Because parameter "wind_turbine_priority" is False, the wind turbine will reduce for the rest.
We make a copy of the microgrid to try it later changing this parameter. 


In [ ]:
import copy

microgrid_copy = copy.deepcopy(microgrid)
microgrid_controller_copy = copy.deepcopy(microgrid_controller)

action = {
    "genset_group": {"status_change": "none"},
    "battery": {"p_grid": 300},
}

safe_action = microgrid_controller.generate_safe_action(action)
microgrid.step(safe_action)
microgrid_controller.update_controller_state({'device_observations': microgrid.gather_observations(), 'controller_state': {}}) 
observations = microgrid_controller.gather_observations()


expected_genset_group_active_power = genset_params['controller']['const_params']['minimum_load']['value']
expected_battery_p_grid = action['battery']['p_grid']
expected_wind_power = observations['device_observations']['demand']['device_observations']['demand'] - genset_params['controller']['const_params']['minimum_load']['value'] - action['battery']['p_grid']
expected_balance = 0

assert np.abs(observations['device_observations']['genset_group']['device_observations']['genset_group_active_power'] - expected_genset_group_active_power) < 10e-2
assert np.abs(observations['device_observations']['battery']['device_observations']['p_grid'] - expected_battery_p_grid) < 10e-2
assert np.abs(observations['device_observations']['wind_turbine']['device_observations']['wind_power'] - expected_wind_power)< 10e-2
assert np.abs(observations['device_observations']['balance'] - expected_balance)< 10e-2

Run with the generator until we pass 30 minutes, then turn it off. The battery will be used to compensate.

In [ ]:
action = {
    "genset_group": {"status_change": "none"},
    "battery": {"p_grid": 0},
}

for i in range(30):
    safe_action = microgrid_controller.generate_safe_action(action)
    microgrid.step(safe_action)
    microgrid_controller.update_controller_state({'device_observations': microgrid.gather_observations(), 'controller_state': {}}) 
    observations = microgrid_controller.gather_observations()

action = {
    "genset_group": {"status_change": "stop_last"},
    "battery": {"p_grid": 0},
}

safe_action = microgrid_controller.generate_safe_action(action)
microgrid.step(safe_action)
microgrid_controller.update_controller_state({'device_observations': microgrid.gather_observations(), 'controller_state': {}}) 
observations = microgrid_controller.gather_observations()

expected_genset_group_active_power = 0
expected_battery_p_grid = observations['device_observations']['demand']['device_observations']['demand'] - observations['device_observations']['wind_turbine']['device_observations']['available_wind_power']
expected_wind_power = observations['device_observations']['wind_turbine']['device_observations']['available_wind_power']
expected_balance = 0

np.testing.assert_allclose(observations['device_observations']['genset_group']['device_observations']['genset_group_active_power'], expected_genset_group_active_power)
np.testing.assert_allclose(observations['device_observations']['battery']['device_observations']['p_grid'], expected_battery_p_grid)
np.testing.assert_allclose(observations['device_observations']['wind_turbine']['device_observations']['wind_power'], expected_wind_power)
np.testing.assert_allclose(observations['device_observations']['balance'], expected_balance)

Run for a bit more. The battery should discharge and the shield will turn on the generator when necessary.

In [ ]:
import matplotlib.pyplot as plt

action = {
    "genset_group": {"status_change": "none"},
    "battery": {"p_grid": 0},
}

battery_socs = []
battery_p_grids = []
balances = []
genset_group_active_powers = []
wind_powers = []
demand = []

for i in range(90):
    safe_action = microgrid_controller.generate_safe_action(action)
    microgrid.step(safe_action)
    microgrid_controller.update_controller_state({'device_observations': microgrid.gather_observations(), 'controller_state': {}}) 
    observations = microgrid_controller.gather_observations()
    
    battery_socs.append(observations['device_observations']['battery']['device_observations']['soc'])
    battery_p_grids.append(observations['device_observations']['battery']['device_observations']['p_grid'])
    balances.append(observations['device_observations']['balance'])
    genset_group_active_powers.append(observations['device_observations']['genset_group']['device_observations']['genset_group_active_power'])
    wind_powers.append(observations['device_observations']['wind_turbine']['device_observations']['wind_power'])
    demand.append(observations['device_observations']['demand']['device_observations']['demand'])
    assert observations['device_observations']['balance'] == 0

plt.figure()
plt.plot(battery_socs)
plt.xlabel("Time (min)")
plt.ylabel("Battery SOC")

plt.figure()
plt.plot(demand)
plt.plot(battery_p_grids)
plt.plot(wind_powers)
plt.plot(genset_group_active_powers)
plt.xlabel("Time (min)")
plt.ylabel("Power (kW)")
plt.legend(["Demand", "Battery", "Wind", "Genset group"])

plt.figure()
plt.plot(balances)
plt.xlabel("Time (min)")
plt.ylabel("Balance (kW)")

safe_action = microgrid_controller.generate_safe_action(action)
microgrid.step(safe_action)
microgrid_controller.update_controller_state({'device_observations': microgrid.gather_observations(), 'controller_state': {}}) 
observations = microgrid_controller.gather_observations()


# In the end, the battery is not used, the wind turbine runs at full power, and the genset compensates
expected_genset_group_active_power = observations['device_observations']['demand']['device_observations']['demand'] - observations['device_observations']['wind_turbine']['device_observations']['available_wind_power']
expected_battery_p_grid = 0
expected_wind_power = observations['device_observations']['wind_turbine']['device_observations']['available_wind_power']
expected_balance = 0

assert np.abs(observations['device_observations']['genset_group']['device_observations']['genset_group_active_power'] - expected_genset_group_active_power) < 10e-2
assert np.abs(observations['device_observations']['battery']['device_observations']['p_grid'] - expected_battery_p_grid) < 10e-2
assert np.abs(observations['device_observations']['wind_turbine']['device_observations']['wind_power'] - expected_wind_power) < 10e-2
assert np.abs(observations['device_observations']['balance'] - expected_balance) < 10e-2

## Wind turbine priority

Now, let's test the wind turbine priority. Let's use the microgrid copy from earlier, change the parameter, and launch the same experience: ask the battery to discharge.
Because of wind_turbine_priority, the battery will discharge but the wind turbine will first be used greedily. The battery command will be overruled.

In [ ]:
microgrid = microgrid_copy
microgrid_controller = microgrid_controller_copy

microgrid_controller.wind_turbine_priority = True      # Changing the configuration parameter

action = {
    "genset_group": {"status_change": "none"},
    "battery": {"p_grid": 300},
}

safe_action = microgrid_controller.generate_safe_action(action)
microgrid.step(safe_action)
microgrid_controller.update_controller_state({'device_observations': microgrid.gather_observations(), 'controller_state': {}}) 
observations = microgrid_controller.gather_observations()

expected_genset_group_active_power = genset_params['controller']['const_params']['minimum_load']['value']
## That is what was expected without wind_turbine_priority
# expected_battery_p_grid = action['battery']['p_grid'] 
# expected_wind_power = observations['device_observations']['demand']['device_observations']['demand'] - genset_params['controller']['const_params']['minimum_load']['value'] - action['battery']['p_grid']

## Now, wind power is done before
expected_wind_power = np.minimum(observations['device_observations']['demand']['device_observations']['demand'] - genset_params['controller']['const_params']['minimum_load']['value'], observations['device_observations']['wind_turbine']['device_observations']['available_wind_power'])
expected_battery_p_grid = observations['device_observations']['demand']['device_observations']['demand'] - genset_params['controller']['const_params']['minimum_load']['value'] - observations['device_observations']['wind_turbine']['device_observations']['wind_power']
expected_balance = 0

np.testing.assert_allclose(observations['device_observations']['genset_group']['device_observations']['genset_group_active_power'], expected_genset_group_active_power)
np.testing.assert_allclose(observations['device_observations']['battery']['device_observations']['p_grid'], expected_battery_p_grid)
np.testing.assert_allclose(observations['device_observations']['wind_turbine']['device_observations']['wind_power'], expected_wind_power)
np.testing.assert_allclose(observations['device_observations']['balance'], expected_balance)

# Test environment

In [ ]:
from env_microgrid import MicroGridEnv
import numpy as np
import datetime
import pathlib

## With an active turbine

In [ ]:
external_world_params = {
    "init_date_time": {"value": datetime.datetime(2018, 6, 6, 6, 6, 0), "type": "datetime"},
    "time_step": {"value": 1, "type": "integer"}
}

genset_params = {
    'device': {
        'const_params': {
            'alpha_g': {'value': 0.25, 'type': 'float'}, 'beta_g': {'value': 10, 'type': 'float'},
            'prime_power_rating': {'value': 400, 'type': 'float'}, 'temp_overload_factor': {'value': 1.1, 'type': 'float'},
            'time_step': {'value': 1, 'type': 'int'}, 'noise_level': {'value': 'none', 'type': 'string'},
            'noise_range_percentage': {'value': 0.2, 'type': 'float'}
        },
        'init_params': {
            'running': {'value': False, 'type': 'bool'}, 'init_power_setpoint': {'value': 300, 'type': 'float'}
        }
    },
    'controller': {
        'const_params': {
            'minimum_run_time': {'value': 30, 'type': 'integer'}, 'minimum_load': {'value': 120, 'type': 'float'},
            'warmup_power': {'value': 100, 'type': 'float'}, 'cooldown_power': {'value': 0, 'type': 'float'},
            'warmup_time': {'value': 3, 'type': 'integer'}, 'cooldown_time': {'value': 6, 'type': 'integer'},
            'average_active_power_constraint': {'value': True, 'type': 'boolean'},
            'average_active_power_window': {'value': 2880, 'type': 'integer'},
            'average_active_power_limit': {'value': 0.7, 'type': 'float'}
        },
        'init_params': {
            'status': {'value': 'off', 'type': 'string', 'elements': ['off', 'running', 'warmup', 'cooldown']},
            'time_since_warmup': {'value': 0, 'type': 'integer'}, 'time_since_cooldown': {'value': 0, 'type': 'integer'},
            'time_since_start': {'value': 0, 'type': 'integer'}, 'average_active_power': {'value': 0, 'type': 'float'}
        }
    }
}

genset_group_params = {
    'device': {
        'const_params': {'n_gensets': {'value': 3, 'type': 'integer'}},
        'init_params': {'gensets': [genset_params] * 3, 'obs_type': 'params'}
    },
    'controller': {
        'const_params': {'priority_order': {'value': True, 'type': 'boolean'}},
        'init_params': {'obs_type': 'params'}
    }
}

battery_params = {
    "device": {
        "const_params": {
            "P_nom": {"value": 600, "type": "float"},
            "soc_min": {"value": 0, "type": "float"},
            "soc_max": {"value": 1, "type": "float"},
            "I_nom": {"value": 672, "type": "float"},
            "Q_max": {"value": 672, "type": "float"},
            "eta_charge": {"value": 0.95, "type": "float"},
            "R": {"value": 0.039375, "type": "float"},
            "A": {"value": 39.6, "type": "float"},
            "B": {"value": 929.16, "type": "float"},
            "alpha_d": {"value": 5, "type": "float"},
            "beta": {"value": 1, "type": "float"},
            "degradation_cost_type": {"value": "cycle_based", "type": "string"},
            "buffer_size": {"value": 30, "type": "integer"},
            "noise_level": {"value": "none", "type": "string"},
            "noise_range_percentage": {"value": 0.2, "type": "float"},
        },
        "init_params": {
            "soc": {"value": 0.5, "type": "float", "min": 0.1, "max": 0.9},
            "p_grid": {"value": 0, "type": "float"},
            "obs_type": 'params'
        },
    },
    "controller": {
        "const_params": {
            "soc_max_norm": {"value": 0.9, "type": "float"},
            "soc_min_norm": {"value": 0.1, "type": "float"},
            "soc_max_res": {"value": 0.95, "type": "float"},
            "soc_min_res": {"value": 0.05, "type": "float"},
        },
        "init_params": {
            "obs_type": 'params'
        },
    },
}


demand_params = {
    'controller': {'const_params':{}, 'init_params':{'obs_type': 'params'}},
    'device': {
        'const_params': {
            "mode": {'value': 'data', 'type': 'string'},
            "pred_time_step": {'value': 1, 'type': 'integer'},
            "nb_pred_time_steps": {'value': 10, 'type': 'integer'},
            "normalisation_factor": {'value': 300, 'type': 'integer'},       # Back to multiply demand by 1        
            "forecast_model": {'value': 'forecast', 'type': 'string'},
        },
        'init_params': {
            "obs_type": 'params',
        }
    }
}


wind_turbine_params = {
    'device': {
        'init_params': {
            'turbine_setpoint': {'value': 300, 'type': 'float'},
            'active': {'value': True, 'type': 'boolean'},
            'obs_type': 'params'
        },
        'const_params': {
            'pred_time_step': {'value': 1, 'type': 'integer'},
            'nb_pred_time_steps': {'value': 10, 'type': 'integer'},
            'mode': {'value': 'data', 'type': 'string'},
            'nominal_power': {'value': 500, 'type': 'float'},
            'perlin_params': {
                'average': {'value': 200, 'type': 'float'},
                'nb_octaves': {'value': 5, 'type': 'integer'},
                'octaves_step': {'value': 4, 'type': 'integer'},
                'period': {'value': 60*24, 'type': 'integer'},
                'persistence': {'value': 0.8, 'type': 'float'},
                'lacunarity': {'value': 2.5, 'type': 'float'},
                'repeat': {'value': 1024, 'type': 'integer'}
            }
        }
    },
    'controller': {
        'init_params': {
            'turbine_setpoint': {'value': 0, 'type': 'float'},
            'obs_type': 'params'
        },
        'const_params': {
            'power_max': {'value': 500, 'type': 'float'},
            'power_min': {'value': 0, 'type': 'float'},
        }
    }
}



microgrid_params = {
    "device": {
        "const_params": {
        },
        "init_params": {
            "external_world": external_world_params,
            "genset_group": genset_group_params,
            "battery": battery_params,
            "demand": demand_params,
            "wind_turbine": wind_turbine_params,
            "obs_type": 'params'
        },
    },
    "controller": {
        "const_params": {
            "check_initialization": {"value": True, "type": "boolean"},
            "init_check_steps": {"value": 8, "type": "integer"},
            "init_check_loops": {"value": 3, "type": "integer"},
            "wind_turbine_priority": {"value": True, "type": "boolean"},
            "check_balance": {"value": True, "type": "boolean"},                       #  Check balance is ON --> Predictive shield on generator status if steady demand in next 10 time step is irrecoverable without reserve
            "check_reserve": {"value": True, "type": "boolean"},                       # Still no check reserve --> No predictive shield on generator status if worst case demand in next 10 time step is irrecoverable with reserve
            "adjust_battery": {"value": True, "type": "boolean"},
            "conservativeness_coeff": {"value": 1, "type": "float"},
            "min_available_wind_power": {"value": 0, "type": "float"},
            "max_available_wind_power_drop_step": {"value": 0.18, "type": "float"},
            "max_demand": {"value": 570, "type": "float"},
            "max_demand_increase_step": {"value": 80, "type": "float"},
        },
        "init_params": {
            "obs_type": 'params',
        },
    }
}

In [ ]:
reward_params = {
    # Fuel consumption
    'fuel_consumption_coeff': 0,
    'controllable_fuel_cons_coeff': 1,
    'configuration_coeff': 0,
    # Maintenance
    'battery_startcharge_coeff': 1,
    'battery_degradation_coeff': 1,
    'battery_variation_coeff': 1,
    'genset_start_coeff': 3,
    'genset_overload_coeff': 0.7,
    # Balance
    'neg_balance_coeff': 100,
    'pos_balance_coeff': 40,
    # Shields
    'battery_shield_coeff': 3,
    'genset_shield_coeff': 3,
    # General control
    'include_fuel_reward' : 1,
    'include_maintenance_reward' : 1,
    'include_shield_reward' : 1,
    'include_balance_reward' : 1,
}
env_params = {
    'microgrid': microgrid_params,
    'reward': reward_params,
    'return_dict': False,
    'train': False,
    'setpoint_action': False,
    'max_episode_steps' : 200000,
    'time_step': 1
}

environment = MicroGridEnv(env_params)


Test a first step

In [ ]:
action = np.array([0.0,0.0])

new_obs, reward, terminated, truncated, info = environment.step(action, verbose = True)

assert new_obs['obs_array'].shape == (15 + 11*genset_group_params['device']['const_params']['n_gensets']['value'],)
assert new_obs['demand_pred_array'].shape == (1, demand_params['device']['const_params']['nb_pred_time_steps']['value'])
assert new_obs['available_wind_power_pred_array'].shape == (1, wind_turbine_params['device']['const_params']['nb_pred_time_steps']['value'] if microgrid_params['device']['init_params']['wind_turbine']['device']['init_params']['active']['value'] else 10)

In [ ]:
new_obs['available_wind_power_pred_array']

In [ ]:
new_obs['demand_pred_array']

### Running time

In [ ]:
import time
action = np.array([0.0,0.0])

# start timer
start_time = time.time()
# Run the environment for 1000 steps
for i in range(10):# range(1000):
    new_obs, reward, terminated, truncated, info = environment.step(action)
    if i%50 == 0:
        print("Step {}".format(i))

# end timer
end_time = time.time()
elapsed_time = end_time - start_time
print("Elapsed time for 1000 steps: {:.2f} seconds".format(elapsed_time))
print("Average time per step: {:.4f} seconds".format(elapsed_time / 1000))

### Resets

In [ ]:
import time
action = np.array([0.0,0.0])

# start timer
start_time = time.time()
# Run the environment for 1000 steps
for i in range(5):
    environment.reset()
    print("Reset {}".format(i))

# end timer
end_time = time.time()
elapsed_time = end_time - start_time
print("Elapsed time for 5 resets: {:.2f} seconds".format(elapsed_time))
print("Average time per reset: {:.4f} seconds".format(elapsed_time / 5))

## Inactive turbine

In [ ]:
env_params['microgrid']['device']['init_params']['wind_turbine']['device']['init_params']['active']['value'] = False

environment = MicroGridEnv(env_params)

In [ ]:
action = np.array([0.0,0.0])

new_obs, reward, terminated, truncated, new_info = environment.step(action, verbose = True)


assert new_obs['obs_array'].shape == (15 + 11*genset_group_params['device']['const_params']['n_gensets']['value'],)
assert new_obs['demand_pred_array'].shape == (1, demand_params['device']['const_params']['nb_pred_time_steps']['value'])
assert new_obs['available_wind_power_pred_array'].shape == (1, wind_turbine_params['device']['const_params']['nb_pred_time_steps']['value'] if microgrid_params['device']['init_params']['wind_turbine']['device']['init_params']['active']['value'] else 10)

## Initialization from parameters and running time

### Initializing from config file

In [ ]:
import yaml
import wandb
config = yaml.safe_load(open("..\\configs\\config.yaml", "r")) 



wandb.init(
    project="RLC-experiments",
    dir="..\\wandb",
    name="env_test",
    config=config,
    sync_tensorboard=True, # keep this true to sync wandb logs
    mode="online"#"disabled" if args.disable_wandb else "online"
)

In [ ]:
from env_microgrid import MicroGridEnv
import numpy as np
import datetime
import pathlib

environment = MicroGridEnv(config['environment'])

### Running time

In [ ]:
import time
action = np.array([0.0,0.0])

# start timer
start_time = time.time()
# Run the environment for 500 steps
for i in range(500):
    new_obs, reward, terminated, truncated, info = environment.step(action)
    if i%50 == 0:
        print("Step {}".format(i))

# end timer
end_time = time.time()
elapsed_time = end_time - start_time
print("Elapsed time for 500 steps: {:.2f} seconds".format(elapsed_time))
print("Average time per step: {:.4f} seconds".format(elapsed_time / 500))

### Reset time

In [ ]:
import time

# start timer
start_time = time.time()
# Reset the environment 500 times
for i in range(500):
    environment.reset()
    if i%50 == 0:
        print("Step {}".format(i))

# end timer
end_time = time.time()
elapsed_time = end_time - start_time
print("Elapsed time for 500 resets: {:.2f} seconds".format(elapsed_time))
print("Average time per reset: {:.4f} seconds".format(elapsed_time / 500))

### Test evaluation environment

In [ ]:
config = yaml.safe_load(open("..\\configs\\config.yaml", "r")) 



eval_environment = MicroGridEnv(config['eval_environment'])


In [ ]:
import time
action = np.array([0.0,0.0])

# start timer
start_time = time.time()
# Run the environment for 100 steps
for i in range(100):
    new_obs, reward, terminated, truncated, info = eval_environment.step(action)
    if i%50 == 0:
        print("Step {}".format(i))

# end timer
end_time = time.time()
elapsed_time = end_time - start_time
print("Elapsed time for 100 steps: {:.2f} seconds".format(elapsed_time))
print("Average time per step: {:.4f} seconds".format(elapsed_time / 100))

In [ ]:
import time

# start timer
start_time = time.time()
# Reset the environment 100 times
for i in range(100):
    eval_environment.reset()
    if i%50 == 0:
        print("Step {}".format(i))

# end timer
end_time = time.time()
elapsed_time = end_time - start_time
print("Elapsed time for 100 resets: {:.2f} seconds".format(elapsed_time))
print("Average time per reset: {:.4f} seconds".format(elapsed_time / 100))